In [ ]:
import base64, hashlib, json, shutil, subprocess, sys
from pathlib import Path
source = Path('/kaggle/working/urop_eval_source')
files = {'model/text_plus_image/bert_deit3/model.py': 'IiIiQkVSVCArIGZpbmUtdHVuZWQgRGVpVCBJSUkgbWF4aW11bSBmdXNpb24sIHNlcGFyYXRlIGZyb20gdGhlIGJhc2VsaW5lLiIiIgojIE1PREVMOiBEZWZpbmVzIEJFUlQgKyBEZWlUIElJSSBtYXhpbXVtLWZlYXR1cmUgZnVzaW9uIGFuZCBzdHJpY3QgRGVpVCBjaGVja3BvaW50IGxvYWRpbmcuCiMgTE9HSUM6IFByZXBhcmVzIGZ1bGwtZnJhbWUgaW1hZ2VzIGFuZCBhcHBsaWVzIHRoZSB2YWxpZGF0aW9uLWJhc2VkIGNoZWNrcG9pbnQgc2VsZWN0aW9uIHJ1bGUuCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgaGFzaGxpYgoKZnJvbSBQSUwgaW1wb3J0IEltYWdlCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKVEVYVF9TSEEyNTYgPSAnODcwYmVhNTBmYTYwOGRkZGQyMDQyMWMyMWZkMTc4NmUyNDRiMTMyM2UyMjczNDA5MzIxNGI4NzAyMzY2Njc1MycKSU1BR0VfU0hBMjU2ID0gJzk3Y2I4OWU5ODZjMjQ4MWQ2NWRkNjFmZjRmMjU3MGE5MDc4OThjODg1YzM0ZWEyZTk0NjAzODVjZTAxYWUzMDAnCklNQUdFX01PREVMID0gJ2RlaXQzX2Jhc2VfcGF0Y2gxNl8yMjQuZmJfaW4yMmtfZnRfaW4xaycKCgpkZWYgc2hhMjU2KHBhdGgpOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogc3RyZWFtLnJlYWQoMTAyNCAqIDEwMjQpLCBiJycpOgogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKCmRlZiBmdWxsX2ZyYW1lKGltYWdlKToKICAgICIiIk1hdGNoIHRoZSA4MyUgaW1hZ2Utb25seSBleHBlcmltZW50LCBpbmNsdWRpbmcgcm91bmRpbmcgYW5kIHBhZGRpbmcuIiIiCiAgICBpbWFnZSA9IGltYWdlLmNvbnZlcnQoJ1JHQicpCiAgICBzY2FsZSA9IDIyNCAvIG1heChpbWFnZS5zaXplKQogICAgZGltcyA9IHR1cGxlKG1heCgxLCBtaW4oMjI0LCByb3VuZChkICogc2NhbGUpKSkgZm9yIGQgaW4gaW1hZ2Uuc2l6ZSkKICAgIHJlc2l6ZWQgPSBpbWFnZS5yZXNpemUoZGltcywgSW1hZ2UuUmVzYW1wbGluZy5CSUxJTkVBUikKICAgIHRpbGUgPSBJbWFnZS5uZXcoJ1JHQicsICgyMjQsIDIyNCksICgxMjQsIDExNiwgMTA0KSkKICAgIHRpbGUucGFzdGUocmVzaXplZCwgKCgyMjQgLSBkaW1zWzBdKSAvLyAyLCAoMjI0IC0gZGltc1sxXSkgLy8gMikpCiAgICByZXR1cm4gdGlsZQoKCmNsYXNzIE1heGltdW1GdXNpb24obm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB0ZXh0LCBpbWFnZSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi50ZXh0LCBzZWxmLmltYWdlID0gdGV4dCwgaW1hZ2UKICAgICAgICBzZWxmLnRleHRfcHJvamVjdGlvbiA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKHRleHQuY29uZmlnLmhpZGRlbl9zaXplLCA1MTIpLCBubi5MYXllck5vcm0oNTEyKSkKICAgICAgICBzZWxmLmltYWdlX3Byb2plY3Rpb24gPSBubi5TZXF1ZW50aWFsKG5uLkxpbmVhcihpbWFnZS5udW1fZmVhdHVyZXMsIDUxMiksIG5uLkxheWVyTm9ybSg1MTIpKQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKDUxMiwgMjU2KSwgbm4uR0VMVSgpLCBubi5Ecm9wb3V0KC4yKSwgbm4uTGluZWFyKDI1NiwgNikpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW5wdXRfaWRzLCBhdHRlbnRpb25fbWFzaywgaW1hZ2VzKToKICAgICAgICB0ZXh0ID0gc2VsZi50ZXh0KGlucHV0X2lkcz1pbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrKS5sYXN0X2hpZGRlbl9zdGF0ZVs6LCAwXQogICAgICAgIGltYWdlID0gc2VsZi5pbWFnZShpbWFnZXMpCiAgICAgICAgcmV0dXJuIHNlbGYuY2xhc3NpZmllcih0b3JjaC5tYXhpbXVtKHNlbGYudGV4dF9wcm9qZWN0aW9uKHRleHQpLCBzZWxmLmltYWdlX3Byb2plY3Rpb24oaW1hZ2UpKSkKCgpkZWYgbG9hZF9pbWFnZV9jaGVja3BvaW50KHBhdGgpOgogICAgIiIiU3RyaWN0bHkgbG9hZCB0aGUgc2l4LWNsYXNzIGltYWdlIGNoZWNrcG9pbnQgYmVmb3JlIHJlbW92aW5nIGl0cyBoZWFkLiIiIgogICAgaW1wb3J0IHRpbW0KICAgIGlmIHNoYTI1NihwYXRoKSAhPSBJTUFHRV9TSEEyNTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW5jb3JyZWN0IGZ1bGwtZnJhbWUgRGVpVCBjaGVja3BvaW50IFNIQS0yNTYnKQogICAgaW1hZ2UgPSB0aW1tLmNyZWF0ZV9tb2RlbChJTUFHRV9NT0RFTCwgcHJldHJhaW5lZD1GYWxzZSkKICAgIGltYWdlLmhlYWQgPSBubi5TZXF1ZW50aWFsKG5uLkRyb3BvdXQoLjIpLCBubi5MaW5lYXIoaW1hZ2UubnVtX2ZlYXR1cmVzLCA2KSkKICAgIHN0YXRlID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249J2NwdScsIHdlaWdodHNfb25seT1UcnVlKQogICAgaW1hZ2UubG9hZF9zdGF0ZV9kaWN0KHN0YXRlLCBzdHJpY3Q9VHJ1ZSkKICAgIGltYWdlLnJlc2V0X2NsYXNzaWZpZXIoMCkKICAgIHJldHVybiBpbWFnZQoKCmRlZiBiZXR0ZXIodmFsdWVzLCBiZXN0KToKICAgICIiIlVucm91bmRlZCBhY2N1cmFjeSwgbWFjcm8tRjEsIGxvc3MsIGFuZCBlYXJsaWVyIGVwb2NoIG9uIHRpZXMuIiIiCiAgICBpZiBiZXN0IGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIFRydWUKICAgIG9sZCA9IGJlc3RbJ21ldHJpY3MnXQogICAgcmV0dXJuICh2YWx1ZXNbJ2FjY3VyYWN5J10sIHZhbHVlc1snbWFjcm9fZjEnXSwgLXZhbHVlc1snbG9zcyddKSA+ICgKICAgICAgICBvbGRbJ2FjY3VyYWN5J10sIG9sZFsnbWFjcm9fZjEnXSwgLW9sZFsnbG9zcyddKQo=', 'model/text_plus_image/bert_deit3/train.py': '...', 'model/text_plus_image/bert_deit3/bert-base-assets/config.json': 'ewogICJhZGRfY3Jvc3NfYXR0ZW50aW9uIjogZmFsc2UsCiAgImFyY2hpdGVjdHVyZXMiOiBbCiAgICAiQmVydEZvck1hc2tlZExNIgogIF0sCiAgImF0dGVudGlvbl9wcm9ic19kcm9wb3V0X3Byb2IiOiAwLjEsCiAgImJvc190b2tlbl9pZCI6IG51bGwsCiAgImNsYXNzaWZpZXJfZHJvcG91dCI6IG51bGwsCiAgImR0eXBlIjogImZsb2F0MzIiLAogICJlb3NfdG9rZW5faWQiOiBudWxsLAogICJncmFkaWVudF9jaGVja3BvaW50aW5nIjogZmFsc2UsCiAgImhpZGRlbl9hY3QiOiAiZ2VsdSIsCiAgImhpZGRlbl9kcm9wb3V0X3Byb2IiOiAwLjEsCiAgImhpZGRlbl9zaXplIjogNzY4LAogICJpZDJsYWJlbCI6IHsKICAgICIwIjogIlRydWUiLAogICAgIjEiOiAiU2F0aXJlL1Bhcm9keSIsCiAgICAiMiI6ICJNaXNsZWFkaW5nIENvbnRlbnQiLAogICAgIjMiOiAiSW1wb3N0ZXIgQ29udGVudCIsCiAgICAiNCI6ICJGYWxzZSBDb25uZWN0aW9uIiwKICAgICI1IjogIk1hbmlwdWxhdGVkIENvbnRlbnQiCiAgfSwKICAiaW5pdGlhbGl6ZXJfcmFuZ2UiOiAwLjAyLAogICJpbnRlcm1lZGlhdGVfc2l6ZSI6IDMwNzIsCiAgImlzX2RlY29kZXIiOiBmYWxzZSwKICAibGFiZWwyaWQiOiB7CiAgICAiRmFsc2UgQ29ubmVjdGlvbiI6IDQsCiAgICAiSW1wb3N0ZXIgQ29udGVudCI6IDMsCiAgICAiTWFuaXB1bGF0ZWQgQ29udGVudCI6IDUsCiAgICAiTWlzbGVhZGluZyBDb250ZW50IjogMiwKICAgICJTYXRpcmUvUGFyb2R5IjogMSwKICAgICJUcnVlIjogMAogIH0sCiAgImxheWVyX25vcm1fZXBzIjogMWUtMTIsCiAgIm1heF9wb3NpdGlvbl9lbWJlZGRpbmdzIjogNTEyLAogICJtb2RlbF90eXBlIjogImJlcnQiLAogICJudW1fYXR0ZW50aW9uX2hlYWRzIjogMTIsCiAgIm51bV9oaWRkZW5fbGF5ZXJzIjogMTIsCiAgInBhZF90b2tlbl9pZCI6IDAsCiAgInBvc2l0aW9uX2VtYmVkZGluZ190eXBlIjogImFic29sdXRlIiwKICAidGllX3dvcmRfZW1iZWRkaW5ncyI6IHRydWUsCiAgInRyYW5zZm9ybWVyc192ZXJzaW9uIjogIjUuMTguMCIsCiAgInR5cGVfdm9jYWJfc2l6ZSI6IDIsCiAgInVzZV9jYWNoZSI6IHRydWUsCiAgInZvY2FiX3NpemUiOiAzMDUyMgp9Cg==', 'model/text_plus_image/bert_deit3/bert-base-assets/tokenizer/tokenizer.json': '...', 'model/text_plus_image/bert_deit3/bert-base-assets/tokenizer/tokenizer_config.json': 'ewogICJiYWNrZW5kIjogInRva2VuaXplcnMiLAogICJjbHNfdG9rZW4iOiAiW0NMU10iLAogICJkb19sb3dlcl9jYXNlIjogdHJ1ZSwKICAiaXNfbG9jYWwiOiBmYWxzZSwKICAibG9jYWxfZmlsZXNfb25seSI6IGZhbHNlLAogICJtYXNrX3Rva2VuIjogIltNQVNLXSIsCiAgIm1vZGVsX21heF9sZW5ndGgiOiA1MTIsCiAgInBhZF90b2tlbiI6ICJbUEFEXSIsCiAgInNlcF90b2tlbiI6ICJbU0VQXSIsCiAgInN0cmlwX2FjY2VudHMiOiBudWxsLAogICJ0b2tlbml6ZV9jaGluZXNlX2NoYXJzIjogdHJ1ZSwKICAidG9rZW5pemVyX2NsYXNzIjogIkJlcnRUb2tlbml6ZXIiLAogICJ1bmtfdG9rZW4iOiAiW1VOS10iCn0K', 'training/text_plus_image/Scripts/usable_cohort_v2.py': 'IiIiUGlubmVkIHVzZXItYXBwcm92ZWQgYXZhaWxhYmxlLWltYWdlIGNvaG9ydDsgb3JpZ2luYWxzIHJlbWFpbiB1bmNoYW5nZWQuCgpUaGlzIHZlcmlmaWVzIHJvdyBtZW1iZXJzaGlwIG9ubHkuIEl0IG5ldmVyIGNlcnRpZmllcyBpbWFnZSBieXRlcyBvciBydW5zIG1vZGVscy4KIiIiCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClJFR0lTVFJBVElPTiA9ICdNTV9QQVJUMV9VU0FCTEVfQ09IT1JUX1JFR0lTVFJBVElPTl8yMDI2MTAyNC5qc29uJwpSRUdJU1RSQVRJT05fU0hBID0gJzNmZTBiNjIyOTU4YTFlZWU2NzMyNzg5ZDRmN2FjZjZiNDUwYzBlM2JkYTg3MzEyZTVjYmI2NTAwZmI0MDVjZTUnCkNPSE9SVCA9ICdGQUNVTFRZLTZXLVVTQUJMRS02ODIxNzAtdjEnClBST1RPQ09MID0gJ09GRklDSUFMLTZXLU1NLVVTQUJMRS12MicKQ09VTlRTID0geyd0cmFpbic6IDU2MzYwMSwgJ3ZhbGlkYXRpb24nOiA1OTI5OSwgJ3Rlc3QnOiA1OTI3MH0KUkVRVUlSRUQgPSA2ODIxNzAKCgpkZWYgc2hhKHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCdyYicpIGFzIGY6CiAgICAgICAgZm9yIGIgaW4gaXRlcihsYW1iZGE6IGYucmVhZCg0ICogMTAyNCoqMiksIGInJyk6CiAgICAgICAgICAgIGgudXBkYXRlKGIpCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiByZWdpc3RyYXRpb24ocGF0aCk6CiAgICBpZiBzaGEocGF0aCkgIT0gUkVHSVNUUkFUSU9OX1NIQToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBcHByb3ZlZCBjb2hvcnQgcmVnaXN0cmF0aW9uIGhhc2ggbWlzbWF0Y2gnKQogICAgdmFsdWUgPSBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KCkpCiAgICBpZiAodmFsdWUuZ2V0KCdhcHByb3ZlZCcpIGlzIG5vdCBUcnVlIG9yIHZhbHVlLmdldCgnY29ob3J0X2lkJykgIT0gQ09IT1JUCiAgICAgICAgICAgIG9yIHZhbHVlLmdldCgncHJvdG9jb2xfaWQnKSAhPSBQUk9UT0NPTCBvciB2YWx1ZS5nZXQoJ3VzYWJsZV9yb3dzJykgIT0gUkVRVUlSRUQKICAgICAgICAgICAgb3IgdmFsdWUuZ2V0KCdzcGxpdF9jb3VudHMnKSAhPSBDT1VOVFMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1dyb25nIGFwcHJvdmVkIGNvaG9ydCcpCiAgICByZXR1cm4gdmFsdWUKCgpkZWYgbG9hZF9jb2hvcnQodHN2X2RpciwgcmVnaXN0cmF0aW9uX3BhdGgsIGV4dHJhY3Rpb25fcGF0aCwgYXBwcm92YWxfcGF0aCk6CiAgICByZWcgPSByZWdpc3RyYXRpb24ocmVnaXN0cmF0aW9uX3BhdGgpCiAgICBpZiBzaGEoYXBwcm92YWxfcGF0aCkgIT0gcmVnWydhcHByb3ZhbF9zaGEyNTYnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVc2VyIGFwcHJvdmFsIHJlY2VpcHQgbWlzbWF0Y2gnKQogICAgaWYganNvbi5sb2FkcyhQYXRoKGFwcHJvdmFsX3BhdGgpLnJlYWRfdGV4dCgpKS5nZXQoJ3N0YXR1cycpICE9ICdhcHByb3ZlZF9ieV91c2VyJzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVc2VyIGFwcHJvdmFsIG1pc3NpbmcnKQogICAgaWYgc2hhKGV4dHJhY3Rpb25fcGF0aCkgIT0gcmVnWydleHRyYWN0aW9uX3JlcG9ydF9zaGEyNTYnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdPcmlnaW5hbCBleHRyYWN0aW9uIHJlcG9ydCBoYXNoIG1pc21hdGNoJykKICAgIHJlcG9ydCA9IGpzb24ubG9hZHMoUGF0aChleHRyYWN0aW9uX3BhdGgpLnJlYWRfdGV4dCgpKQogICAgaWYgKHJlcG9ydC5nZXQoJ3NvdXJjZV9pZCcpICE9IHJlZ1snYXJjaGl2ZV9zb3VyY2VfaWQnXQogICAgICAgICAgICBvciByZXBvcnQuZ2V0KCdjb21wcmVzc2VkX2FyY2hpdmVfc2hhMjU2JykgIT0gcmVnWydhcmNoaXZlX3NoYTI1NiddCiAgICAgICAgICAgIG9yIHJlcG9ydC5nZXQoJ2NvbXByZXNzZWRfYXJjaGl2ZV9ieXRlcycpICE9IHJlZ1snYXJjaGl2ZV9ieXRlcyddCiAgICAgICAgICAgIG9yIHJlcG9ydC5nZXQoJ3NvdXJjZV9jcmNfY29uc3VtZWQnKSBpcyBub3QgVHJ1ZQogICAgICAgICAgICBvciByZXBvcnQuZ2V0KCdyZXF1aXJlZCcpICE9IHJlZ1snb3JpZ2luYWxfcm93cyddCiAgICAgICAgICAgIG9yIHJlcG9ydC5nZXQoJ3VzYWJsZScpICE9IFJFUVVJUkVECiAgICAgICAgICAgIG9yIHJlcG9ydC5nZXQoJ2ltYWdlX2dhdGVfcGFzc2VkJykgaXMgbm90IEZhbHNlKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdPcmlnaW5hbCBhbGwtcm93IHNvdXJjZSBldmlkZW5jZSBkaWZmZXJzOyBuZXZlciByZXdyaXRlIGl0cyBmYWlsZWQgZ2F0ZScpCiAgICBmYWlsdXJlcyA9IHJlcG9ydFsnZGVjb2RlX2ZhaWx1cmVzJ10KICAgIGV4Y2x1ZGVkID0ge3Jvd1snaWQnXSBmb3Igcm93IGluIGZhaWx1cmVzfQogICAgZmluZ2VycHJpbnQgPSBoYXNobGliLnNoYTI1NigoJ1xuJy5qb2luKHNvcnRlZChleGNsdWRlZCkpICsgJ1xuJykuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCiAgICBpZiAobGVuKGZhaWx1cmVzKSAhPSBsZW4oZXhjbHVkZWQpIG9yIGxlbihleGNsdWRlZCkgIT0gcmVnWydleGNsdWRlZF9yb3dzJ10KICAgICAgICAgICAgb3Igc2V0KHJlcG9ydFsnbWlzc2luZ19pZHMnXSkgIT0gZXhjbHVkZWQgb3IgZmluZ2VycHJpbnQgIT0gcmVnWydleGNsdWRlZF9pZHNfc2hhMjU2J10pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4Y2x1c2lvbnMgZGlmZmVyIGZyb20gZXhhY3QgYXBwcm92ZWQgNDkxIElEcycpCiAgICBhbGxfaWRzLCBmb3VuZCwgZXhwZWN0ZWQgPSBzZXQoKSwgc2V0KCksIHt9CiAgICBjb3VudHMsIGZhaWxlZF9jb3VudHMsIGZhaWxlZF9jbGFzc2VzID0gQ291bnRlcigpLCBDb3VudGVyKCksIENvdW50ZXIoKQogICAgY2xhc3NlcyA9IHtrZXk6IENvdW50ZXIoKSBmb3Iga2V5IGluIENPVU5UU30KICAgIGZvciBpdGVtIGluIHJlZ1snb3JpZ2luYWxfZmlsZXMnXToKICAgICAgICBwYXRoID0gUGF0aCh0c3ZfZGlyKSAvIGl0ZW1bJ2ZpbGVuYW1lJ10KICAgICAgICBpZiBzaGEocGF0aCkgIT0gaXRlbVsnc2hhMjU2J106CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ09yaWdpbmFsIGZhY3VsdHkgVFNWIGhhc2ggbWlzbWF0Y2g6ICcgKyBpdGVtWydmaWxlbmFtZSddKQogICAgICAgIGNvdW50ID0gMAogICAgICAgIHdpdGggcGF0aC5vcGVuKG5ld2xpbmU9JycpIGFzIHN0cmVhbToKICAgICAgICAgICAgZm9yIHJvdyBpbiBjc3YuRGljdFJlYWRlcihzdHJlYW0sIGRlbGltaXRlcj0nXHQnKToKICAgICAgICAgICAgICAgIGlkZW50ID0gcm93WydpZCddOyBsYWJlbCA9IGludChyb3dbJzZfd2F5X2xhYmVsJ10pOyBjb3VudCArPSAxCiAgICAgICAgICAgICAgICBpZiBpZGVudCBpbiBhbGxfaWRzIG9yIG5vdCBpZGVudC5pc2FsbnVtKCkgb3IgbGFiZWwgbm90IGluIHJhbmdlKDYpIG9yIG5vdCByb3dbJ2NsZWFuX3RpdGxlJ10uc3RyaXAoKToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkL2R1cGxpY2F0ZSBvcmlnaW5hbCByb3cnKQogICAgICAgICAgICAgICAgYWxsX2lkcy5hZGQoaWRlbnQpCiAgICAgICAgICAgICAgICBzcGxpdCA9IGl0ZW1bJ3NwbGl0J10KICAgICAgICAgICAgICAgIGlmIGlkZW50IGluIGV4Y2x1ZGVkOgogICAgICAgICAgICAgICAgICAgIGZvdW5kLmFkZChpZGVudCk7IGZhaWxlZF9jb3VudHNbc3BsaXRdICs9IDE7IGZhaWxlZF9jbGFzc2VzW3N0cihsYWJlbCldICs9IDEKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgZXhwZWN0ZWRbaWRlbnRdID0geydpZCc6IGlkZW50LCAnc3BsaXQnOiBzcGxpdCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2NsZWFuX3RpdGxlJzogcm93WydjbGVhbl90aXRsZSddLCAnNl93YXlfbGFiZWwnOiBsYWJlbH0KICAgICAgICAgICAgICAgICAgICBjb3VudHNbc3BsaXRdICs9IDE7IGNsYXNzZXNbc3BsaXRdW3N0cihsYWJlbCldICs9IDEKICAgICAgICBpZiBjb3VudCAhPSBpdGVtWydyb3dzJ106CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ09yaWdpbmFsIHJvdyBjb3VudCBtaXNtYXRjaCcpCiAgICBpZiAobGVuKGFsbF9pZHMpICE9IHJlZ1snb3JpZ2luYWxfcm93cyddIG9yIGZvdW5kICE9IGV4Y2x1ZGVkIG9yIGxlbihleHBlY3RlZCkgIT0gUkVRVUlSRUQKICAgICAgICAgICAgb3IgZGljdChjb3VudHMpICE9IENPVU5UUyBvciBkaWN0KGZhaWxlZF9jb3VudHMpICE9IHJlZ1snZXhjbHVkZWRfc3BsaXRfY291bnRzJ10KICAgICAgICAgICAgb3IgZGljdChmYWlsZWRfY2xhc3NlcykgIT0gcmVnWydleGNsdWRlZF9jbGFzc19jb3VudHMnXQogICAgICAgICAgICBvciB7a2V5OiBkaWN0KHZhbHVlKSBmb3Iga2V5LCB2YWx1ZSBpbiBjbGFzc2VzLml0ZW1zKCl9ICE9IHJlZ1snY2xhc3NfY291bnRzJ10pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FwcHJvdmVkIGNvaG9ydCBtZW1iZXJzaGlwL2NvdW50L2xhYmVsIGF1ZGl0IGZhaWxlZCcpCiAgICByZXR1cm4gZXhwZWN0ZWQsIHJlZwoKCmRlZiBsb2FkX2lucHV0KHJvb3QpOgogICAgcm9vdCA9IFBhdGgocm9vdCkKICAgIHJlZyA9IHJlZ2lzdHJhdGlvbihyb290IC8gJ3Jlc3VsdHMnIC8gUkVHSVNUUkFUSU9OKQogICAgcmV0dXJuIGxvYWRfY29ob3J0KHJvb3QgLyAnZGF0YS9vZmZpY2lhbF9tdWx0aW1vZGFsX3YyJywgcm9vdCAvICdyZXN1bHRzJyAvIFJFR0lTVFJBVElPTiwKICAgICAgICAgICAgICAgICAgICAgICByb290IC8gJ3Jlc3VsdHMnIC8gcmVnWydleHRyYWN0aW9uX3JlcG9ydF9maWxlJ10sCiAgICAgICAgICAgICAgICAgICAgICAgcm9vdCAvICdyZXN1bHRzJyAvIHJlZ1snYXBwcm92YWxfZmlsZSddKQo=', 'training/text_plus_image/Scripts/multimodal_recovery_v1.py': 'IiIiUmVjb3ZlcnkgcHJpbWl0aXZlcyBmb3IgT0ZGSUNJQUwtNlctTU0tdjE7IG5vdCBhIGZ1bGwgdHJhaW5lci9sYXVuY2ggZ2F0ZS4KCk5vIG5ldHdvcmsgb3IgR1BVIGFsbG9jYXRpb24uIFN0YXRlIHNlcmlhbGl6YXRpb24gdHJ1c3RzIG9ubHkgb3duZWQgbG9jYWwgZmlsZXMuCiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJhbmRvbQppbXBvcnQgdXVpZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNodmlzaW9uIGltcG9ydCB0cmFuc2Zvcm1zIGFzIFQKClNDSEVNQT0nTU0tUkVDT1ZFUlktdjEnCgpkZWYgc2hhKHBhdGgpOgogaD1oYXNobGliLnNoYTI1NigpCiB3aXRoIG9wZW4ocGF0aCwncmInKSBhcyBmOgogIGZvciBiIGluIGl0ZXIobGFtYmRhOmYucmVhZCgxMDI0KjEwMjQpLGInJyk6aC51cGRhdGUoYikKIHJldHVybiBoLmhleGRpZ2VzdCgpCgpkZWYgc3RhYmxlX3NlZWQoc2VlZCxlcG9jaCxpZGVudCk6CiByZXR1cm4gaW50LmZyb21fYnl0ZXMoaGFzaGxpYi5zaGEyNTYoanNvbi5kdW1wcyhbU0NIRU1BLHNlZWQsZXBvY2gsc3RyKGlkZW50KV0sc2VwYXJhdG9ycz0oJywnLCc6JykpLmVuY29kZSgpKS5kaWdlc3QoKVs6OF0sJ2JpZycpJSgyKio2My0xKQoKZGVmIG9yZGVyKHNlZWQsZXBvY2gsY291bnQpOgogcmV0dXJuIHRvcmNoLnJhbmRwZXJtKGNvdW50LGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzdGFibGVfc2VlZChzZWVkLGVwb2NoLCdzYW1wbGVyJykpKQoKZGVmIGNyb3AoaW1hZ2Usc2VlZCxlcG9jaCxpZGVudCk6CiAjIExvY2FsIENQVSBSTkcgZm9yayBtYWtlcyBhdWdtZW50YXRpb24gaW5kZXBlbmRlbnQgb2Ygd29ya2VyL21vZGVsIFJORy4KIHdpdGggdG9yY2gucmFuZG9tLmZvcmtfcm5nKGRldmljZXM9W10pOgogICMgdG9yY2gubWFudWFsX3NlZWQgYWxzbyBzZWVkcyBDVURBOyBjcm9wIG11c3QgbmV2ZXIgY2hhbmdlIG1vZGVsIENVREEgUk5HLgogIHRvcmNoLnJhbmRvbS5kZWZhdWx0X2dlbmVyYXRvci5tYW51YWxfc2VlZChzdGFibGVfc2VlZChzZWVkLGVwb2NoLGlkZW50KSkKICByZXR1cm4gVC5SYW5kb21SZXNpemVkQ3JvcC5nZXRfcGFyYW1zKGltYWdlLCguOCwxLiksKC43NSw0LzMpKQoKZGVmIGNhcHR1cmVfcm5nKCk6CiByZXR1cm4geydweXRob24nOnJhbmRvbS5nZXRzdGF0ZSgpLCdudW1weSc6bnAucmFuZG9tLmdldF9zdGF0ZSgpLCd0b3JjaCc6dG9yY2guZ2V0X3JuZ19zdGF0ZSgpLAogICAgICAgICAnY3VkYSc6dG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlX2FsbCgpIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSBbXX0KCmRlZiByZXN0b3JlX3JuZyhzdGF0ZSk6CiByYW5kb20uc2V0c3RhdGUoc3RhdGVbJ3B5dGhvbiddKTtucC5yYW5kb20uc2V0X3N0YXRlKHN0YXRlWydudW1weSddKTt0b3JjaC5zZXRfcm5nX3N0YXRlKHN0YXRlWyd0b3JjaCddLmNwdSgpKQogaWYgc3RhdGVbJ2N1ZGEnXTp0b3JjaC5jdWRhLnNldF9ybmdfc3RhdGVfYWxsKFt4LmNwdSgpIGZvciB4IGluIHN0YXRlWydjdWRhJ11dKQoKZGVmIG1hcF90ZXh0X2VuY29kZXIoc291cmNlLHRhcmdldCk6CiBleGNsdWRlZD17J2JlcnQucG9vbGVyLmRlbnNlLndlaWdodCcsJ2JlcnQucG9vbGVyLmRlbnNlLmJpYXMnLCdjbGFzc2lmaWVyLndlaWdodCcsJ2NsYXNzaWZpZXIuYmlhcyd9CiBpZiBzZXQoc291cmNlKS1leGNsdWRlZCAhPSB7J2JlcnQuJytrIGZvciBrIGluIHRhcmdldC5zdGF0ZV9kaWN0KCl9OnJhaXNlIFZhbHVlRXJyb3IoJ1VuZXhwZWN0ZWQgd2FybS1zdGFydCBrZXlzJykKIG1hcHBlZD17ay5yZW1vdmVwcmVmaXgoJ2JlcnQuJyk6diBmb3Igayx2IGluIHNvdXJjZS5pdGVtcygpIGlmIGsgbm90IGluIGV4Y2x1ZGVkfQogZm9yIGssdiBpbiB0YXJnZXQuc3RhdGVfZGljdCgpLml0ZW1zKCk6CiAgaWYgdi5zaGFwZSE9bWFwcGVkW2tdLnNoYXBlIG9yIHYuZHR5cGUhPW1hcHBlZFtrXS5kdHlwZTpyYWlzZSBWYWx1ZUVycm9yKCdXYXJtLXN0YXJ0IHNoYXBlL2R0eXBlICcraykKIHRhcmdldC5sb2FkX3N0YXRlX2RpY3QobWFwcGVkLHN0cmljdD1UcnVlKQogcmV0dXJuIHsnbWFwcGVkX2tleXMnOmxlbihtYXBwZWQpLCdleGNsdWRlZF9rZXlzJzpzb3J0ZWQoZXhjbHVkZWQpLCdzdHJpY3QnOlRydWV9CgpkZWYgYXRvbWljX2pzb24ocGF0aCx2YWx1ZSk6CiBwYXRoPVBhdGgocGF0aCk7dG1wPXBhdGgud2l0aF9uYW1lKHBhdGgubmFtZSsnLnRtcC0nK3V1aWQudXVpZDQoKS5oZXgpCiB3aXRoIHRtcC5vcGVuKCd3JykgYXMgZjpqc29uLmR1bXAodmFsdWUsZixpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpO2Yud3JpdGUoJ1xuJyk7Zi5mbHVzaCgpO29zLmZzeW5jKGYuZmlsZW5vKCkpCiBvcy5yZXBsYWNlKHRtcCxwYXRoKQoKZGVmIHNhdmVfZ2VuZXJhdGlvbihmb2xkZXIscGF5bG9hZCk6CiBmb2xkZXI9UGF0aChmb2xkZXIpO2ZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKIGlmIHBheWxvYWQuZ2V0KCdzY2hlbWEnKSE9U0NIRU1BIG9yIG5vdCBwYXlsb2FkLmdldCgnZ3JhZGllbnRzX2NsZWFyZWQnKTpyYWlzZSBWYWx1ZUVycm9yKCdOb3QgYSBzYWZlIHJlY292ZXJ5IGJvdW5kYXJ5JykKIGdlbmVyYXRpb249dXVpZC51dWlkNCgpLmhleDtwYXRoPWZvbGRlci8oZ2VuZXJhdGlvbisnLnB0Jyk7dG1wPXBhdGgud2l0aF9zdWZmaXgoJy50bXAnKQogd2l0aCB0bXAub3Blbignd2InKSBhcyBmOnRvcmNoLnNhdmUocGF5bG9hZCxmKTtmLmZsdXNoKCk7b3MuZnN5bmMoZi5maWxlbm8oKSkKIG9zLnJlcGxhY2UodG1wLHBhdGgpCiAjIFJlbG9hZCBiZWZvcmUgY29tbWl0dGluZyBwb2ludGVyLiBUaGlzIGFsb25lIGlzIHNjcmF0Y2ggcGVyc2lzdGVuY2UsIG5vdCBjbG91ZCBkdXJhYmlsaXR5LgogY2hlY2s9dG9yY2gubG9hZChwYXRoLG1hcF9sb2NhdGlvbj0nY3B1Jyx3ZWlnaHRzX29ubHk9RmFsc2UpCiBpZiBjaGVja1snc2NoZW1hJ10hPVNDSEVNQSBvciBjaGVja1snY3Vyc29yJ10hPXBheWxvYWRbJ2N1cnNvciddOnJhaXNlIFZhbHVlRXJyb3IoJ1JlbG9hZCBmYWlsZWQnKQogaXRlbT17J2ZpbGUnOnBhdGgubmFtZSwnc2hhMjU2JzpzaGEocGF0aCksJ2J5dGVzJzpwYXRoLnN0YXQoKS5zdF9zaXplLCdjdXJzb3InOnBheWxvYWRbJ2N1cnNvciddfQogcG9pbnRlcj1mb2xkZXIvJ2dlbmVyYXRpb25zLmpzb24nO29sZD1qc29uLmxvYWRzKHBvaW50ZXIucmVhZF90ZXh0KCkpIGlmIHBvaW50ZXIuZXhpc3RzKCkgZWxzZSB7J2dlbmVyYXRpb25zJzpbXX0KIGdlbmVyYXRpb25zPVtpdGVtXStvbGRbJ2dlbmVyYXRpb25zJ11bOjFdO2F0b21pY19qc29uKHBvaW50ZXIseydzY2hlbWEnOlNDSEVNQSwnZ2VuZXJhdGlvbnMnOmdlbmVyYXRpb25zfSkKICMgQm91bmQgc2NyYXRjaCBzcGFjZSBvbmx5IGFmdGVyIHRoZSBuZXcgcG9pbnRlciBpcyBjb21taXR0ZWQuIE5ldmVyIHN3ZWVwCiAjIHVucmVsYXRlZCBmaWxlcyBvciB1bmNvbW1pdHRlZC9jb3JydXB0IGFydGlmYWN0cyByZXRhaW5lZCBmb3IgZGlhZ25vc2lzLgogZm9yIHJldGlyZWQgaW4gb2xkWydnZW5lcmF0aW9ucyddWzE6XToKICBjYW5kaWRhdGU9Zm9sZGVyL3JldGlyZWRbJ2ZpbGUnXQogIGlmIGNhbmRpZGF0ZS5wYXJlbnQucmVzb2x2ZSgpPT1mb2xkZXIucmVzb2x2ZSgpIGFuZCBsZW4oY2FuZGlkYXRlLnN0ZW0pPT0zMiBhbmQgYWxsKGMgaW4gJzAxMjM0NTY3ODlhYmNkZWYnIGZvciBjIGluIGNhbmRpZGF0ZS5zdGVtKToKICAgY2FuZGlkYXRlLnVubGluayhtaXNzaW5nX29rPVRydWUpCiByZXR1cm4gaXRlbQoKZGVmIGxvYWRfZ2VuZXJhdGlvbihmb2xkZXIpOgogZm9sZGVyPVBhdGgoZm9sZGVyKTtwb2ludGVyPWpzb24ubG9hZHMoKGZvbGRlci8nZ2VuZXJhdGlvbnMuanNvbicpLnJlYWRfdGV4dCgpKTtlcnJvcnM9W10KIGZvciBnZW5lcmF0aW9uIGluIHBvaW50ZXJbJ2dlbmVyYXRpb25zJ106CiAgdHJ5OgogICBwYXRoPWZvbGRlci9nZW5lcmF0aW9uWydmaWxlJ10KICAgaWYgcGF0aC5wYXJlbnQucmVzb2x2ZSgpIT1mb2xkZXIucmVzb2x2ZSgpIG9yIHNoYShwYXRoKSE9Z2VuZXJhdGlvblsnc2hhMjU2J106cmFpc2UgVmFsdWVFcnJvcignSGFzaC9wYXRoIG1pc21hdGNoJykKICAgcGF5bG9hZD10b3JjaC5sb2FkKHBhdGgsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1GYWxzZSkKICAgaWYgcGF5bG9hZFsnc2NoZW1hJ10hPVNDSEVNQSBvciBwYXlsb2FkWydjdXJzb3InXSE9Z2VuZXJhdGlvblsnY3Vyc29yJ106cmFpc2UgVmFsdWVFcnJvcignU2NoZW1hL2N1cnNvciBtaXNtYXRjaCcpCiAgIHJldHVybiBwYXlsb2FkLHsnbG9hZGVkJzpnZW5lcmF0aW9uLCdmYWxsYmFja19lcnJvcnMnOmVycm9yc30KICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixSdW50aW1lRXJyb3IsS2V5RXJyb3IpIGFzIGU6ZXJyb3JzLmFwcGVuZChzdHIoZSkpCiByYWlzZSBWYWx1ZUVycm9yKCdObyB1c2FibGUgcmVjb3ZlcnkgZ2VuZXJhdGlvbjogJytyZXByKGVycm9ycykpCg==', 'training/text_plus_image/Scripts/packed_image_io_v3.py': 'IiIiTG9zc2xlc3MgVEFSIGltYWdlIHN0b3JhZ2UgYWRhcHRlci4gRG9lcyBub3QgY2hhbmdlIHBpeGVscyBvciByZXNlYXJjaCByb3dzLgoKQnVpbGQgYSBib3VuZGVkIGluZGV4IHdoaWxlIHN0cmljdGx5IGhhc2hpbmcvZGVjb2RpbmcgZXZlcnkgbWVtYmVyLiBSZWFkIG9ubHkKdGhlIHBpbm5lZCBieXRlIHJhbmdlIGR1cmluZyB0cmFpbmluZy4gTm8gaW1hZ2UgZXh0cmFjdGlvbiBvciBtb2RlbCBleGVjdXRpb24uCiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aCwgUHVyZVBvc2l4UGF0aAppbXBvcnQgc3FsaXRlMwppbXBvcnQgdGFyZmlsZQppbXBvcnQgc2h1dGlsCmltcG9ydCB0aW1lCmZyb20gY29uY3VycmVudC5mdXR1cmVzIGltcG9ydCBUaHJlYWRQb29sRXhlY3V0b3IKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgT3JkZXJlZERpY3QKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpfSU5ERVhfQ0FDSEUgPSB7fQpfQVJDSElWRV9DQUNIRSA9IHt9CgoKZGVmIGNhY2hlX3Jhd19hcmNoaXZlcyhyb290LCBwcm9vZnMsIGNhY2hlLCBtaW5pbXVtX2ZyZWU9MTIwICogMTAyNCoqMyk6CiAgICAiIiJDb3B5IGV4YWN0IHNvdXJjZSBhcmNoaXZlcyBpbnRvIGJvdW5kZWQgY2xvdWQgc2NyYXRjaDsgbmV2ZXIgcmUtZW5jb2RlLgoKICAgIEEgY29tcGxldGUgU0hBIHByb29mLCBub3QgYSBmaWxlbmFtZSBvciBwcmlvciByZWNlaXB0LCBhdXRob3JpemVzIHJldXNlLgogICAgT3JpZ2luYWxzIHJlbWFpbiByZWFkLW9ubHkuIFRlbXBvcmFyeSB3cml0ZXMgYXJlIG93bmVkIGJ5IHRoaXMgY2FjaGUgb25seS4KICAgICIiIgogICAgcm9vdCwgY2FjaGUgPSBQYXRoKHJvb3QpLnJlc29sdmUoKSwgUGF0aChjYWNoZSkucmVzb2x2ZSgpCiAgICB0b3RhbCA9IHN1bShzaXplIGZvciBzaXplLCBkaWdlc3QgaW4gcHJvb2ZzLnZhbHVlcygpKQogICAgaWYgbm90IHByb29mcyBvciB0b3RhbCA+IDEwMCAqIDEwMjQqKjM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUmF3IGFyY2hpdmUgY2FjaGUgZXhjZWVkcyByZWdpc3RlcmVkIHN0b3JhZ2UgYm91bmQnKQogICAgY2FjaGUubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgaWYgc2h1dGlsLmRpc2tfdXNhZ2UoY2FjaGUpLmZyZWUgPCBtaW5pbXVtX2ZyZWU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW5zdWZmaWNpZW50IGNsb3VkIHNjcmF0Y2ggZm9yIHZlcmlmaWVkIHJhdyBhcmNoaXZlIGNhY2hlJykKICAgIHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCgogICAgZGVmIGNvcHlfb25lKGl0ZW0pOgogICAgICAgIG5hbWUsIChzaXplLCBkaWdlc3QpID0gaXRlbQogICAgICAgIHNvdXJjZSwgdGFyZ2V0ID0gcm9vdCAvIG5hbWUsIGNhY2hlIC8gbmFtZQogICAgICAgIGlmIHNvdXJjZS5wYXJlbnQgIT0gcm9vdCBvciB0YXJnZXQucGFyZW50ICE9IGNhY2hlIG9yIG5vdCBuYW1lLmVuZHN3aXRoKCcuYmluJyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1Vuc2FmZSByYXcgYXJjaGl2ZSBjYWNoZSBuYW1lJykKICAgICAgICBpZiBzb3VyY2Uuc3RhdCgpLnN0X3NpemUgIT0gc2l6ZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignU291cmNlIHJhdyBhcmNoaXZlIHNpemUgY2hhbmdlZCcpCiAgICAgICAgaWYgdGFyZ2V0LmV4aXN0cygpOgogICAgICAgICAgICBpZiB0YXJnZXQuc3RhdCgpLnN0X3NpemUgIT0gc2l6ZSBvciBzaGEodGFyZ2V0KSAhPSBkaWdlc3Q6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdQcmVzZXJ2ZSBjb25mbGljdGluZyBjbG91ZCByYXcgYXJjaGl2ZSBjYWNoZScpCiAgICAgICAgICAgIHJldHVybiB7J2ZpbGUnOiBuYW1lLCAnYnl0ZXMnOiBzaXplLCAnc2hhMjU2JzogZGlnZXN0LCAncmV1c2VkJzogVHJ1ZX0KICAgICAgICB0ZW1wb3JhcnkgPSB0YXJnZXQud2l0aF9zdWZmaXgoJy5jb3B5LScgKyBzdHIob3MuZ2V0cGlkKCkpKQogICAgICAgIGlmIHRlbXBvcmFyeS5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUHJlc2VydmUgaW50ZXJydXB0ZWQgcmF3IGFyY2hpdmUgY29weSBmb3IgaW5zcGVjdGlvbicpCiAgICAgICAgaCwgY291bnQgPSBoYXNobGliLnNoYTI1NigpLCAwCiAgICAgICAgd2l0aCBzb3VyY2Uub3BlbigncmInKSBhcyBzcmMsIHRlbXBvcmFyeS5vcGVuKCd4YicpIGFzIGRzdDoKICAgICAgICAgICAgd2hpbGUgYmxvY2sgOj0gc3JjLnJlYWQoMTYgKiAxMDI0KioyKToKICAgICAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkID4gMzYwMDoKICAgICAgICAgICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoJ1JhdyBhcmNoaXZlIHN0YWdpbmcgZXhjZWVkZWQgb25lLWhvdXIgYm91bmQnKQogICAgICAgICAgICAgICAgZHN0LndyaXRlKGJsb2NrKTsgaC51cGRhdGUoYmxvY2spOyBjb3VudCArPSBsZW4oYmxvY2spCiAgICAgICAgICAgIGRzdC5mbHVzaCgpOyBvcy5mc3luYyhkc3QuZmlsZW5vKCkpCiAgICAgICAgaWYgY291bnQgIT0gc2l6ZSBvciBoLmhleGRpZ2VzdCgpICE9IGRpZ2VzdDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUmF3IGFyY2hpdmUgY2FjaGUgY29weSBTSEEgbWlzbWF0Y2gnKQogICAgICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCB0YXJnZXQpCiAgICAgICAgcHJpbnQoanNvbi5kdW1wcyh7J3N0YXRlJzogJ3Jhd19hcmNoaXZlX2NhY2hlZCcsICdmaWxlJzogbmFtZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAnYnl0ZXMnOiBjb3VudCwgJ3NlY29uZHMnOiB0aW1lLm1vbm90b25pYygpLXN0YXJ0ZWR9KSwgZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4geydmaWxlJzogbmFtZSwgJ2J5dGVzJzogY291bnQsICdzaGEyNTYnOiBkaWdlc3QsICdyZXVzZWQnOiBGYWxzZX0KCiAgICB3aXRoIFRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz00KSBhcyBwb29sOgogICAgICAgIGNvcGllZCA9IGxpc3QocG9vbC5tYXAoY29weV9vbmUsIHNvcnRlZChwcm9vZnMuaXRlbXMoKSkpKQogICAgcmVjZWlwdCA9IHsnc3RhdGUnOiAnZXhhY3RfcmF3X2FyY2hpdmVfY2FjaGVfdmVyaWZpZWQnLCAnc291cmNlJzogc3RyKHJvb3QpLAogICAgICAgICAgICAgICAnY2FjaGUnOiBzdHIoY2FjaGUpLCAnYnl0ZXMnOiB0b3RhbCwgJ2FyY2hpdmVzJzogY29waWVkLAogICAgICAgICAgICAgICAnc2Vjb25kcyc6IHRpbWUubW9ub3RvbmljKCktc3RhcnRlZCwgJ3BpeGVsc19jaGFuZ2VkJzogRmFsc2V9CiAgICAoY2FjaGUgLyAncmF3X2FyY2hpdmVfY2FjaGVfcmVjZWlwdC5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY2VpcHQsIGluZGVudD0yKSsnXG4nKQogICAgcmV0dXJuIGNhY2hlCgoKZGVmIHNoYShwYXRoKToKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigncmInKSBhcyBzdHJlYW06CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCg0ICogMTAyNCoqMiksIGInJyk6CiAgICAgICAgICAgIGgudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgcGFja2VkX25hbWUoYXJjaGl2ZSk6CiAgICBwID0gUHVyZVBvc2l4UGF0aChhcmNoaXZlKQogICAgaWYgbGVuKHAucGFydHMpICE9IDEgb3Igbm90IGFyY2hpdmUuZW5kc3dpdGgoJy50YXInKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdFeHBlY3RlZCBhIHNpbXBsZSBvcmlnaW5hbCBUQVIgZmlsZW5hbWUnKQogICAgcmV0dXJuIGFyY2hpdmVbOi00XSArICcuYmluJwoKCmNsYXNzIEhhc2hSZWFkZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgc3RyZWFtKToKICAgICAgICBzZWxmLnN0cmVhbSwgc2VsZi5kaWdlc3QsIHNlbGYuY291bnQgPSBzdHJlYW0sIGhhc2hsaWIuc2hhMjU2KCksIDAKCiAgICBkZWYgcmVhZChzZWxmLCBzaXplPS0xKToKICAgICAgICBibG9jayA9IHNlbGYuc3RyZWFtLnJlYWQoc2l6ZSkKICAgICAgICBzZWxmLmRpZ2VzdC51cGRhdGUoYmxvY2spCiAgICAgICAgc2VsZi5jb3VudCArPSBsZW4oYmxvY2spCiAgICAgICAgcmV0dXJuIGJsb2NrCgoKZGVmIGluZGV4X2FyY2hpdmUocm9vdCwgYXJjaGl2ZSwgcHJvb2YsIHJvd3MsIGNvbm5lY3Rpb24pOgogICAgIiIiT25lIHRyYW5zYWN0aW9uIHBlciBhcmNoaXZlOyBjYWxsZXIgY29tbWl0cyBhIGZpbmFsIGluZGV4IG9ubHkgYWZ0ZXIgYWxsLiIiIgogICAgcGF0aCA9IFBhdGgocm9vdCkgLyBwYWNrZWRfbmFtZShhcmNoaXZlKQogICAgaWYgcGF0aC5zdGF0KCkuc3Rfc2l6ZSAhPSBwcm9vZlsnYnl0ZXMnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdQYWNrZWQgYXJjaGl2ZSBzaXplIG1pc21hdGNoJykKICAgIGV4cGVjdGVkID0ge3JbJ2lkJ106IHIgZm9yIHIgaW4gcm93c30KICAgIGlmIGxlbihleHBlY3RlZCkgIT0gbGVuKHJvd3MpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBleHBlY3RlZCBpbWFnZSBJRHMnKQogICAgc2VlbiA9IHNldCgpCiAgICB3aXRoIHBhdGgub3BlbigncmInKSBhcyBzb3VyY2U6CiAgICAgICAgcmVhZGVyID0gSGFzaFJlYWRlcihzb3VyY2UpCiAgICAgICAgd2l0aCB0YXJmaWxlLm9wZW4oZmlsZW9iaj1yZWFkZXIsIG1vZGU9J3J8JykgYXMgY29udGFpbmVyOgogICAgICAgICAgICBmb3IgbWVtYmVyIGluIGNvbnRhaW5lcjoKICAgICAgICAgICAgICAgIHAgPSBQdXJlUG9zaXhQYXRoKG1lbWJlci5uYW1lKQogICAgICAgICAgICAgICAgaWYgbm90IG1lbWJlci5pc2ZpbGUoKSBvciBwLmlzX2Fic29sdXRlKCkgb3IgJy4uJyBpbiBwLnBhcnRzOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1VuZXhwZWN0ZWQgbm9uLWltYWdlIG9yIHVuc2FmZSBUQVIgbWVtYmVyJykKICAgICAgICAgICAgICAgIGlkZW50ID0gcC5zdGVtCiAgICAgICAgICAgICAgICByb3cgPSBleHBlY3RlZC5nZXQoaWRlbnQpCiAgICAgICAgICAgICAgICBpZiByb3cgaXMgTm9uZSBvciBpZGVudCBpbiBzZWVuIG9yIHJvd1snc291cmNlX3NoYXJkJ10gIT0gYXJjaGl2ZToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbmV4cGVjdGVkL2R1cGxpY2F0ZSBpbWFnZSBtZW1iZXJzaGlwJykKICAgICAgICAgICAgICAgIGlmIHN0cihQdXJlUG9zaXhQYXRoKFBhdGgoYXJjaGl2ZSkuc3RlbSkgLyBwKSAhPSByb3dbJ2ltYWdlX3BhdGgnXToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdUQVIgbWVtYmVyIGRpZmZlcnMgZnJvbSBvcmlnaW5hbCBwYWlyZWQgbWFuaWZlc3QnKQogICAgICAgICAgICAgICAgaWYgbWVtYmVyLnNpemUgIT0gaW50KHJvd1snYnl0ZXMnXSk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVEFSIGltYWdlIHNpemUgZGlmZmVycyBmcm9tIG9yaWdpbmFsIHBhaXJlZCBtYW5pZmVzdCcpCiAgICAgICAgICAgICAgICByYXcgPSBjb250YWluZXIuZXh0cmFjdGZpbGUobWVtYmVyKS5yZWFkKCkKICAgICAgICAgICAgICAgIGlmIGxlbihyYXcpICE9IG1lbWJlci5zaXplIG9yIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkgIT0gcm93WydpbWFnZV9zaGEyNTYnXToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdQYWNrZWQgaW1hZ2UgU0hBIG1pc21hdGNoOiAnICsgaWRlbnQpCiAgICAgICAgICAgICAgICB3aXRoIEltYWdlLm9wZW4oaW8uQnl0ZXNJTyhyYXcpKSBhcyBpbWFnZToKICAgICAgICAgICAgICAgICAgICBpbWFnZS5sb2FkKCkKICAgICAgICAgICAgICAgICAgICBpbWFnZS5jb252ZXJ0KCdSR0InKS5sb2FkKCkKICAgICAgICAgICAgICAgIGNvbm5lY3Rpb24uZXhlY3V0ZSgnSU5TRVJUIElOVE8gaW1hZ2VzIFZBTFVFUyAoPyw/LD8sPyw/KScsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKGlkZW50LCBwYWNrZWRfbmFtZShhcmNoaXZlKSwgbWVtYmVyLm9mZnNldF9kYXRhLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtZW1iZXIuc2l6ZSwgcm93WydpbWFnZV9zaGEyNTYnXSkpCiAgICAgICAgICAgICAgICBzZWVuLmFkZChpZGVudCkKICAgICAgICAjIEhhc2ggcmVtYWluaW5nIFRBUiBwYWRkaW5nIHRvbywgbm90IG1lcmVseSB0aGUgbGFzdCBpbWFnZSBwYXlsb2FkLgogICAgICAgIHdoaWxlIHJlYWRlci5yZWFkKDQgKiAxMDI0KioyKToKICAgICAgICAgICAgcGFzcwogICAgICAgIGlmIHJlYWRlci5jb3VudCAhPSBwcm9vZlsnYnl0ZXMnXSBvciByZWFkZXIuZGlnZXN0LmhleGRpZ2VzdCgpICE9IHByb29mWydzaGEyNTYnXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRnVsbCBvcmlnaW5hbCBUQVIgU0hBIG1pc21hdGNoJykKICAgIGlmIHNlZW4gIT0gc2V0KGV4cGVjdGVkKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbmNvbXBsZXRlIGFyY2hpdmUgaW1hZ2UgbWVtYmVyc2hpcCcpCiAgICByZXR1cm4gbGVuKHNlZW4pCgoKZGVmIGNyZWF0ZV9pbmRleChyb290LCBzaGFyZHMsIHJvd3MsIG91dHB1dCk6CiAgICBvdXRwdXQgPSBQYXRoKG91dHB1dCkKICAgIGlmIG91dHB1dC5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdQcmVzZXJ2ZSBhbiBleGlzdGluZyBpbmRleDsgZXhwbGljaXQgcmVjb3ZlcnkgcmVxdWlyZWQnKQogICAgb3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0ZW1wb3JhcnkgPSBvdXRwdXQud2l0aF9zdWZmaXgoJy5idWlsZGluZy5zcWxpdGUnKQogICAgZ3JvdXBzID0ge30KICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICBncm91cHMuc2V0ZGVmYXVsdChyb3dbJ3NvdXJjZV9zaGFyZCddLCBbXSkuYXBwZW5kKHJvdykKICAgIGlmIHNldChncm91cHMpICE9IHtzWydmaWxlJ10gZm9yIHMgaW4gc2hhcmRzfToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBcmNoaXZlIHNldCBkaWZmZXJzIGZyb20gbWFuaWZlc3QnKQogICAgY291bnQgPSAwCiAgICB3aXRoIHNxbGl0ZTMuY29ubmVjdCh0ZW1wb3JhcnkpIGFzIGRiOgogICAgICAgIGRiLmV4ZWN1dGUoJ1BSQUdNQSBzeW5jaHJvbm91cz1GVUxMJykKICAgICAgICBpZiBub3QgZGIuZXhlY3V0ZSgiU0VMRUNUIG5hbWUgRlJPTSBzcWxpdGVfbWFzdGVyIFdIRVJFIHR5cGU9J3RhYmxlJyBBTkQgbmFtZT0naW1hZ2VzJyIpLmZldGNob25lKCk6CiAgICAgICAgICAgIGRiLmV4ZWN1dGUoJ0NSRUFURSBUQUJMRSBpbWFnZXMgKGlkIFRFWFQgUFJJTUFSWSBLRVksIGFyY2hpdmUgVEVYVCBOT1QgTlVMTCwgb2Zmc2V0IElOVEVHRVIgTk9UIE5VTEwsIHNpemUgSU5URUdFUiBOT1QgTlVMTCwgc2hhMjU2IFRFWFQgTk9UIE5VTEwpJykKICAgICAgICAgICAgZGIuZXhlY3V0ZSgnQ1JFQVRFIFRBQkxFIGFyY2hpdmVzIChuYW1lIFRFWFQgUFJJTUFSWSBLRVksIGJ5dGVzIElOVEVHRVIgTk9UIE5VTEwsIHNoYTI1NiBURVhUIE5PVCBOVUxMLCBpbWFnZXMgSU5URUdFUiBOT1QgTlVMTCknKQogICAgICAgICAgICBkYi5jb21taXQoKQogICAgICAgIGlmIHt4WzBdIGZvciB4IGluIGRiLmV4ZWN1dGUoIlNFTEVDVCBuYW1lIEZST00gc3FsaXRlX21hc3RlciBXSEVSRSB0eXBlPSd0YWJsZSciKX0gIT0geydpbWFnZXMnLCAnYXJjaGl2ZXMnfToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVW5yZWNvZ25pemVkIGluY29tcGxldGUgaW5kZXggc2NoZW1hOyBwcmVzZXJ2ZSBmb3IgaW5zcGVjdGlvbicpCiAgICAgICAgaWYgZGIuZXhlY3V0ZSgnUFJBR01BIGludGVncml0eV9jaGVjaycpLmZldGNob25lKClbMF0gIT0gJ29rJzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW5jb21wbGV0ZSBTUUxpdGUgaW5kZXggZmFpbGVkIGludGVncml0eSBjaGVjaycpCiAgICAgICAgY29tcGxldGVkID0ge25hbWU6IChzaXplLCBkaWdlc3QsIG4pIGZvciBuYW1lLCBzaXplLCBkaWdlc3QsIG4gaW4KICAgICAgICAgICAgICAgICAgICAgZGIuZXhlY3V0ZSgnU0VMRUNUIG5hbWUsYnl0ZXMsc2hhMjU2LGltYWdlcyBGUk9NIGFyY2hpdmVzJyl9CiAgICAgICAgaWYgc2V0KGNvbXBsZXRlZCkgLSBzZXQoZ3JvdXBzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW5jb21wbGV0ZSBpbmRleCBjb250YWlucyBhbiB1bmV4cGVjdGVkIGFyY2hpdmUnKQogICAgICAgIGZvciBzaGFyZCBpbiBzaGFyZHM6CiAgICAgICAgICAgIG5hbWUgPSBzaGFyZFsnZmlsZSddCiAgICAgICAgICAgIGlmIG5hbWUgaW4gY29tcGxldGVkOgogICAgICAgICAgICAgICAgc2l6ZSwgZGlnZXN0LCBuID0gY29tcGxldGVkW25hbWVdCiAgICAgICAgICAgICAgICBhY3R1YWwgPSBsaXN0KGRiLmV4ZWN1dGUoJ1NFTEVDVCBpZCxvZmZzZXQsc2l6ZSxzaGEyNTYgRlJPTSBpbWFnZXMgV0hFUkUgYXJjaGl2ZT0/JywgKHBhY2tlZF9uYW1lKG5hbWUpLCkpKQogICAgICAgICAgICAgICAgZXhwZWN0ZWQgPSB7clsnaWQnXTogKGludChyWydieXRlcyddKSwgclsnaW1hZ2Vfc2hhMjU2J10pIGZvciByIGluIGdyb3Vwc1tuYW1lXX0KICAgICAgICAgICAgICAgIGlmIChzaXplLCBkaWdlc3QsIG4pICE9IChzaGFyZFsnYnl0ZXMnXSwgc2hhcmRbJ3NoYTI1NiddLCBsZW4oZXhwZWN0ZWQpKSBvciBsZW4oYWN0dWFsKSAhPSBuOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0luY29tcGxldGUgaW5kZXggYXJjaGl2ZSBwcm9vZiBkaWZmZXJzIGZyb20gcGlubmVkIHNvdXJjZScpCiAgICAgICAgICAgICAgICBpZiBhbnkoaWRlbnQgbm90IGluIGV4cGVjdGVkIG9yIG9mZnNldCA8IDAgb3IgKGxlbmd0aCwgaW1hZ2Vfc2hhKSAhPSBleHBlY3RlZFtpZGVudF0KICAgICAgICAgICAgICAgICAgICAgICBmb3IgaWRlbnQsIG9mZnNldCwgbGVuZ3RoLCBpbWFnZV9zaGEgaW4gYWN0dWFsKToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbmNvbXBsZXRlIGluZGV4IGltYWdlIHJvd3MgZGlmZmVyIGZyb20gcGlubmVkIG1hbmlmZXN0JykKICAgICAgICAgICAgICAgIHBhdGggPSBQYXRoKHJvb3QpIC8gcGFja2VkX25hbWUobmFtZSkKICAgICAgICAgICAgICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemUgIT0gc2l6ZSBvciBzaGEocGF0aCkgIT0gZGlnZXN0OgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1ByZXZpb3VzbHkgaW5kZXhlZCBhcmNoaXZlIGJ5dGVzIGNoYW5nZWQnKQogICAgICAgICAgICAgICAgY291bnQgKz0gbgogICAgICAgICAgICAgICAgcHJpbnQoanNvbi5kdW1wcyh7J3N0YXRlJzogJ3ByZXZpb3VzX3BhY2tlZF9hcmNoaXZlX3JldmVyaWZpZWQnLCAnYXJjaGl2ZSc6IG5hbWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAndmVyaWZpZWRfaW1hZ2VzJzogY291bnR9KSwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGRiLmV4ZWN1dGUoJ0JFR0lOJykKICAgICAgICAgICAgICAgIG4gPSBpbmRleF9hcmNoaXZlKHJvb3QsIG5hbWUsIHNoYXJkLCBncm91cHNbbmFtZV0sIGRiKQogICAgICAgICAgICAgICAgZGIuZXhlY3V0ZSgnSU5TRVJUIElOVE8gYXJjaGl2ZXMgVkFMVUVTICg/LD8sPyw/KScsIChuYW1lLCBzaGFyZFsnYnl0ZXMnXSwgc2hhcmRbJ3NoYTI1NiddLCBuKSkKICAgICAgICAgICAgICAgIGRiLmNvbW1pdCgpCiAgICAgICAgICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgZGIucm9sbGJhY2soKQogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgY291bnQgKz0gbgogICAgICAgICAgICBwcmludChqc29uLmR1bXBzKHsnc3RhdGUnOiAncGFja2VkX2FyY2hpdmVfZnVsbHlfdmVyaWZpZWQnLCAnYXJjaGl2ZSc6IHNoYXJkWydmaWxlJ10sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICd2ZXJpZmllZF9pbWFnZXMnOiBjb3VudH0pLCBmbHVzaD1UcnVlKQogICAgICAgIGFzc2VydCBkYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKikgRlJPTSBpbWFnZXMnKS5mZXRjaG9uZSgpWzBdID09IGNvdW50ID09IGxlbihyb3dzKQogICAgd2l0aCB0ZW1wb3Jhcnkub3BlbigncmInKSBhcyBzdHJlYW06CiAgICAgICAgb3MuZnN5bmMoc3RyZWFtLmZpbGVubygpKQogICAgb3MucmVwbGFjZSh0ZW1wb3JhcnksIG91dHB1dCkKICAgIHJldHVybiB7J2ltYWdlcyc6IGNvdW50LCAnaW5kZXhfc2hhMjU2Jzogc2hhKG91dHB1dCksICdpbmRleCc6IHN0cihvdXRwdXQpfQoKCmNsYXNzIEFyY2hpdmVTdG9yZToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCByb290LCBpbmRleCwgaW5kZXhfc2hhMjU2KToKICAgICAgICBzZWxmLnJvb3QsIHNlbGYuc291cmNlX2luZGV4ID0gUGF0aChyb290KS5yZXNvbHZlKCksIFBhdGgoaW5kZXgpLnJlc29sdmUoKQogICAgICAgIGlmIHNoYShzZWxmLnNvdXJjZV9pbmRleCkgIT0gaW5kZXhfc2hhMjU2OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBcmNoaXZlIGluZGV4IFNIQSBtaXNtYXRjaCcpCiAgICAgICAgc2VsZi5pbmRleCA9IHNlbGYuc291cmNlX2luZGV4CiAgICAgICAgIyBLYWdnbGUgaW5wdXRzIHVzZSBhIHJlbW90ZSBmaWxlc3lzdGVtLiBTUUxpdGUncyB0aW55IHJhbmRvbSBwYWdlIHJlYWRzCiAgICAgICAgIyB3ZXJlIHRoZSBtZWFzdXJlZCBib3R0bGVuZWNrOyBjYWNoZSBvbmx5IHRoaXMgZXhhY3QgMTA4IE1CIGluZGV4LgogICAgICAgIGlmIHNlbGYuc291cmNlX2luZGV4LmlzX3JlbGF0aXZlX3RvKFBhdGgoJy9rYWdnbGUvaW5wdXQnKSk6CiAgICAgICAgICAgIGNhY2hlID0gUGF0aCgnL2thZ2dsZS93b3JraW5nL3Vyb3BfdmVyaWZpZWRfaW5kZXhfY2FjaGUnKQogICAgICAgICAgICBjYWNoZS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIGxvY2FsID0gY2FjaGUgLyAoaW5kZXhfc2hhMjU2ICsgJy5zcWxpdGUnKQogICAgICAgICAgICBpZiBub3QgbG9jYWwuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICB0ZW1wb3JhcnkgPSBsb2NhbC53aXRoX3N1ZmZpeCgnLnRtcC0nICsgc3RyKG9zLmdldHBpZCgpKSkKICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZShzZWxmLnNvdXJjZV9pbmRleCwgdGVtcG9yYXJ5KQogICAgICAgICAgICAgICAgaWYgc2hhKHRlbXBvcmFyeSkgIT0gaW5kZXhfc2hhMjU2OgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0NvcGllZCBpbmRleCBTSEEgbWlzbWF0Y2gnKQogICAgICAgICAgICAgICAgb3MucmVwbGFjZSh0ZW1wb3JhcnksIGxvY2FsKQogICAgICAgICAgICBpZiBzaGEobG9jYWwpICE9IGluZGV4X3NoYTI1NjoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1ByZXNlcnZlIGNvbmZsaWN0aW5nIGNhY2hlZCBpbmRleCcpCiAgICAgICAgICAgIHNlbGYuaW5kZXggPSBsb2NhbAogICAgICAgIGtleSA9IChzdHIoc2VsZi5pbmRleCksIGluZGV4X3NoYTI1NikKICAgICAgICBpZiBrZXkgbm90IGluIF9JTkRFWF9DQUNIRToKICAgICAgICAgICAgd2l0aCBzcWxpdGUzLmNvbm5lY3QoJ2ZpbGU6JyArIHN0cihzZWxmLmluZGV4KSArICc/bW9kZT1ybyZpbW11dGFibGU9MScsIHVyaT1UcnVlKSBhcyBkYjoKICAgICAgICAgICAgICAgIHJvd3MgPSB7aWRlbnQ6IChuYW1lLCBvZmZzZXQsIHNpemUsIGRpZ2VzdCkgZm9yIGlkZW50LCBuYW1lLCBvZmZzZXQsIHNpemUsIGRpZ2VzdCBpbgogICAgICAgICAgICAgICAgICAgICAgICBkYi5leGVjdXRlKCdTRUxFQ1QgaWQsYXJjaGl2ZSxvZmZzZXQsc2l6ZSxzaGEyNTYgRlJPTSBpbWFnZXMnKX0KICAgICAgICAgICAgICAgIHByb29mcyA9IHtwYWNrZWRfbmFtZShuYW1lKTogKHNpemUsIGRpZ2VzdCkgZm9yIG5hbWUsIHNpemUsIGRpZ2VzdCBpbiBkYi5leGVjdXRlKCdTRUxFQ1QgbmFtZSxieXRlcyxzaGEyNTYgRlJPTSBhcmNoaXZlcycpfQogICAgICAgICAgICBfSU5ERVhfQ0FDSEVba2V5XSA9IHJvd3MsIHByb29mcwogICAgICAgIHNlbGYucm93cywgcHJvb2ZzID0gX0lOREVYX0NBQ0hFW2tleV0KICAgICAgICBzZWxmLnNpemVzID0ge25hbWU6IGl0ZW1bMF0gZm9yIG5hbWUsIGl0ZW0gaW4gcHJvb2ZzLml0ZW1zKCl9CiAgICAgICAgaWYgc2VsZi5yb290LmlzX3JlbGF0aXZlX3RvKFBhdGgoJy9rYWdnbGUvaW5wdXQnKSk6CiAgICAgICAgICAgIGNhY2hlX2tleSA9IChzdHIoc2VsZi5yb290KSwgaW5kZXhfc2hhMjU2KQogICAgICAgICAgICBpZiBjYWNoZV9rZXkgbm90IGluIF9BUkNISVZFX0NBQ0hFOgogICAgICAgICAgICAgICAgX0FSQ0hJVkVfQ0FDSEVbY2FjaGVfa2V5XSA9IGNhY2hlX3Jhd19hcmNoaXZlcygKICAgICAgICAgICAgICAgICAgICBzZWxmLnJvb3QsIHByb29mcywgJy90bXAvdXJvcF92ZXJpZmllZF9yYXdfYXJjaGl2ZXMvJyArIGluZGV4X3NoYTI1NikKICAgICAgICAgICAgc2VsZi5yb290ID0gX0FSQ0hJVkVfQ0FDSEVbY2FjaGVfa2V5XQogICAgICAgIHNlbGYucGlkLCBzZWxmLmRiLCBzZWxmLmhhbmRsZXMgPSBOb25lLCBOb25lLCBPcmRlcmVkRGljdCgpCgogICAgZGVmIF9fZ2V0c3RhdGVfXyhzZWxmKToKICAgICAgICByZXR1cm4geyoqc2VsZi5fX2RpY3RfXywgJ3BpZCc6IE5vbmUsICdkYic6IE5vbmUsICdoYW5kbGVzJzogT3JkZXJlZERpY3QoKX0KCiAgICBkZWYgcmVhZChzZWxmLCByb3cpOgogICAgICAgIGlmIHNlbGYucGlkICE9IG9zLmdldHBpZCgpOgogICAgICAgICAgICBmb3IgaGFuZGxlIGluIHNlbGYuaGFuZGxlcy52YWx1ZXMoKToKICAgICAgICAgICAgICAgIGhhbmRsZS5jbG9zZSgpCiAgICAgICAgICAgIHNlbGYuaGFuZGxlcywgc2VsZi5waWQgPSBPcmRlcmVkRGljdCgpLCBvcy5nZXRwaWQoKQogICAgICAgIGl0ZW0gPSBzZWxmLnJvd3MuZ2V0KHJvd1snaWQnXSkKICAgICAgICBpZiBpdGVtIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ltYWdlIG1pc3NpbmcgZnJvbSB2ZXJpZmllZCBhcmNoaXZlIGluZGV4JykKICAgICAgICBuYW1lLCBvZmZzZXQsIHNpemUsIGRpZ2VzdCA9IGl0ZW0KICAgICAgICBpZiBuYW1lICE9IHBhY2tlZF9uYW1lKHJvd1snc291cmNlX3NoYXJkJ10pIG9yIHNpemUgIT0gaW50KHJvd1snYnl0ZXMnXSkgb3IgZGlnZXN0ICE9IHJvd1snaW1hZ2Vfc2hhMjU2J106CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FyY2hpdmUgaW5kZXggZGlmZmVycyBmcm9tIHBpbm5lZCBvcmlnaW5hbCBtYW5pZmVzdCcpCiAgICAgICAgaWYgbmFtZSBub3QgaW4gc2VsZi5zaXplcyBvciBvZmZzZXQgPCAwIG9yIG9mZnNldCArIHNpemUgPiBzZWxmLnNpemVzW25hbWVdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGluZGV4ZWQgYXJjaGl2ZSBieXRlIHJhbmdlJykKICAgICAgICBpZiBuYW1lIG5vdCBpbiBzZWxmLmhhbmRsZXM6CiAgICAgICAgICAgIHBhdGggPSAoc2VsZi5yb290IC8gbmFtZSkucmVzb2x2ZSgpCiAgICAgICAgICAgIGlmIHBhdGgucGFyZW50ICE9IHNlbGYucm9vdCBvciBwYXRoLnN0YXQoKS5zdF9zaXplICE9IHNlbGYuc2l6ZXNbbmFtZV06CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdQYWNrZWQgYXJjaGl2ZSBwYXRoL3NpemUgY2hhbmdlZCcpCiAgICAgICAgICAgIHNlbGYuaGFuZGxlc1tuYW1lXSA9IHBhdGgub3BlbigncmInLCBidWZmZXJpbmc9MCkKICAgICAgICAgICAgaWYgbGVuKHNlbGYuaGFuZGxlcykgPiA2NDoKICAgICAgICAgICAgICAgIF8sIG9sZCA9IHNlbGYuaGFuZGxlcy5wb3BpdGVtKGxhc3Q9RmFsc2UpCiAgICAgICAgICAgICAgICBvbGQuY2xvc2UoKQogICAgICAgIHN0cmVhbSA9IHNlbGYuaGFuZGxlcy5wb3AobmFtZSkKICAgICAgICBzZWxmLmhhbmRsZXNbbmFtZV0gPSBzdHJlYW0KICAgICAgICBzdHJlYW0uc2VlayhvZmZzZXQpCiAgICAgICAgcmF3ID0gc3RyZWFtLnJlYWQoc2l6ZSkKICAgICAgICBpZiBsZW4ocmF3KSAhPSBzaXplIG9yIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkgIT0gZGlnZXN0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbmRleGVkIGltYWdlIGJ5dGVzIGNoYW5nZWQnKQogICAgICAgIHJldHVybiByYXcK', 'configs/text_plus_image/bert_deit3/experiment.json': 'ewogICJwcm90b2NvbF9pZCI6ICJPRkZJQ0lBTC02Vy1NTS1VU0FCTEUtdjIiLAogICJzZWVkIjogNDIsCiAgInBsYW5uZWRfZXBvY2hzIjogNCwKICAiYmVydF9jaGVja3BvaW50IjogImdvb2dsZS1iZXJ0L2JlcnQtYmFzZS11bmNhc2VkIiwKICAiYmVydF9yZXZpc2lvbiI6ICI4NmI1ZTA5MzQ0OTRiZDE1Yzk2MzJiMTJmNzM0YThhNjdmNzIzNTk0IiwKICAid2FybXN0YXJ0X3NoYTI1NiI6ICI4NzBiZWE1MGZhNjA4ZGRkZDIwNDIxYzIxZmQxNzg2ZTI0NGIxMzIzZTIyNzM0MDkzMjE0Yjg3MDIzNjY2NzUzIiwKICAibWF4X2xlbmd0aCI6IDEyOCwKICAibWljcm9fYmF0Y2giOiA4LAogICJhY2N1bXVsYXRpb24iOiA0LAogICJldmFsX2JhdGNoIjogMzIsCiAgIndvcmtlcnMiOiAyLAogICJiZXJ0X2xyIjogMWUtMDUsCiAgImltYWdlX2xyIjogM2UtMDUsCiAgImhlYWRfbHIiOiAwLjAwMSwKICAiYmVydF9oZWFkX2RlY2F5IjogMC4wMSwKICAiaW1hZ2VfZGVjYXkiOiAwLjAwMDEsCiAgIndhcm11cF9mcmFjdGlvbiI6IDAuMSwKICAiZ3JhZGllbnRfY2xpcCI6IDEuMCwKICAiY2hlY2twb2ludF91cGRhdGVzIjogMTAwMCwKICAiY2hlY2twb2ludF9zZWNvbmRzIjogNjAwLAogICJlYXJseV9zdG9wX21pbl9lcG9jaCI6IDUsCiAgImVhcmx5X3N0b3BfcGF0aWVuY2UiOiAyLAogICJlYXJseV9zdG9wX2RlbHRhIjogMC4wMDEsCiAgImRldGVybWluaXNtIjogIndhcm4tb25seTsgc3RhYmxlIHNlZWQvZXBvY2gvaWQgQ1BVIGNyb3A7IGV4cGxpY2l0IGVwb2NoIHBlcm11dGF0aW9uIiwKICAic2VsZWN0aW9uIjogInZhbGlkYXRpb24gYWNjdXJhY3ksIG1hY3JvLUYxLCBsb3dlciBsb3NzLCBlYXJsaWVyIGVwb2NoIiwKICAidGVzdF9wb2xpY3kiOiAib25lIGZyb3plbiBoZWxkLW91dCBldmFsdWF0aW9uIG9ubHkiLAogICJmcmVlemVfcG9saWN5IjogIlVzZXItYXBwcm92ZWR1c2FibGUtaW1hZ2Vjb2hvcnQ7c2NpZW50aWZpY3J1bmZyZWV6ZXNvbmx5YWZ0ZXJyZWFsZGF0YS9jbG91ZHByZWZsaWdodHBhc3NlcyIsCiAgImNvaG9ydF9pZCI6ICJGQUNVTFRZLTZXLVVTQUJMRS02ODIxNzAtdjEiLAogICJjb2hvcnRfcmVnaXN0cmF0aW9uX3NoYTI1NiI6ICIzZmUwYjYyMjk1OGExZWVlNjczMjc4OWQ0ZjdhY2Y2YjQ1MGMwZTNiZGE4NzMxMmU1Y2JiNjUwMGZiNDA1Y2U1IiwKICAidHJhaW5fcm93cyI6IDU2MzYwMSwKICAidmFsaWRhdGlvbl9yb3dzIjogNTkyOTksCiAgInRlc3Rfcm93c19pbnRlZ3JpdHlfb25seSI6IDU5MjcwLAogICJwbGFubmVkX3VwZGF0ZV9ib3VuZGFyaWVzIjogNzA0NTIsCiAgIndhcm11cF9ib3VuZGFyaWVzIjogNzA0NiwKICAiZXhwZXJpbWVudF9pZCI6ICJCRVJULURFSVQzLUZVTExGUkFNRS12MSIsCiAgImltYWdlX21vZGVsIjogImRlaXQzX2Jhc2VfcGF0Y2gxNl8yMjQuZmJfaW4yMmtfZnRfaW4xayIsCiAgImltYWdlX3dhcm1zdGFydF9zaGEyNTYiOiAiOTdjYjg5ZTk4NmMyNDgxZDY1ZGQ2MWZmNGYyNTcwYTkwNzg5OGM4ODVjMzRlYTJlOTQ2MDM4NWNlMDFhZTMwMCIsCiAgImltYWdlX3ByZXByb2Nlc3NpbmciOiAiMjI0IGZ1bGwtZnJhbWUgYmlsaW5lYXIgcmVzaXplIGFuZCBjZW50ZXJlZCBSR0IgKDEyNCwxMTYsMTA0KSBwYWRkaW5nOyBJbWFnZU5ldCBub3JtYWxpemF0aW9uIiwKICAic3RhdHVzIjogInByb3Zpc2lvbmFsOyBmdWxsIEdQVSBwcmVmbGlnaHQgYW5kIHF1b3RhIGdhdGUgcmVxdWlyZWQiLAogICJlYXJseV9zdG9wX3BvbGljeSI6ICJkaXNhYmxlZCBmb3IgdGhlIGZvdXIgcGxhbm5lZCBlcG9jaHMiCn0K', 'testing/text_plus_image/verify_multimodal_final_v1.py': 'IiIiSW5kZXBlbmRlbnQgQ1NWLW9ubHkgdmVyaWZpY2F0aW9uIGZvciBhIGZyb3plbiBtdWx0aW1vZGFsIHZhbGlkYXRpb24vdGVzdC4KTm8gbW9kZWwsIGluZmVyZW5jZSwgdHJhaW5pbmcgb3IgbmV0d29yayBhY2Nlc3MuIE91dHB1dCBpcyBhIHJldHVybmVkIGRpY3QuCiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGhhc2hsaWIsanNvbgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYWNjdXJhY3lfc2NvcmUsZjFfc2NvcmUsYmFsYW5jZWRfYWNjdXJhY3lfc2NvcmUsY2xhc3NpZmljYXRpb25fcmVwb3J0LGNvbmZ1c2lvbl9tYXRyaXgKTEFCRUxTPVsnVHJ1ZScsJ1NhdGlyZS9QYXJvZHknLCdNaXNsZWFkaW5nIENvbnRlbnQnLCdJbXBvc3RlciBDb250ZW50JywnRmFsc2UgQ29ubmVjdGlvbicsJ01hbmlwdWxhdGVkIENvbnRlbnQnXQpkZWYgc2hhKHBhdGgpOgogaD1oYXNobGliLnNoYTI1NigpCiB3aXRoIFBhdGgocGF0aCkub3BlbigncmInKSBhcyBmOgogIGZvciBiIGluIGl0ZXIobGFtYmRhOmYucmVhZCg0KjEwMjQqKjIpLGInJyk6aC51cGRhdGUoYikKIHJldHVybiBoLmhleGRpZ2VzdCgpCmRlZiB2ZXJpZnkoZm9sZGVyLHByZWZpeCxvZmZpY2lhbCxzcGxpdCk6CiBmb2xkZXI9UGF0aChmb2xkZXIpO2ZyYW1lPXBkLnJlYWRfY3N2KGZvbGRlci8ocHJlZml4KydfcHJlZGljdGlvbnMuY3N2JyksZHR5cGU9eydpZCc6c3RyfSkKIGFzc2VydCBsZW4oZnJhbWUpPT1sZW4ob2ZmaWNpYWwpIGFuZCBmcmFtZS5pZC5pc191bmlxdWUgYW5kIGZyYW1lLmlkLnRvbGlzdCgpPT1vZmZpY2lhbC5pZC50b2xpc3QoKQogYXNzZXJ0IGZyYW1lWydzcGxpdCddLmVxKHNwbGl0KS5hbGwoKQogeT1mcmFtZS50cnVlX2xhYmVsLnRvX251bXB5KCk7cD1mcmFtZS5wcmVkaWN0ZWRfbGFiZWwudG9fbnVtcHkoKTtjZT1mcmFtZS5jcm9zc19lbnRyb3B5LnRvX251bXB5KCk7cHJvYnM9ZnJhbWVbWydwJytzdHIoaSkgZm9yIGkgaW4gcmFuZ2UoNildXS50b19udW1weSgpCiBhc3NlcnQgbnAuYXJyYXlfZXF1YWwoeSxvZmZpY2lhbFsnNl93YXlfbGFiZWwnXS5hc3R5cGUoaW50KS50b19udW1weSgpKQogYXNzZXJ0IG5wLmlzZmluaXRlKHByb2JzKS5hbGwoKSBhbmQgbnAuaXNmaW5pdGUoY2UpLmFsbCgpIGFuZCAoY2U+PTApLmFsbCgpIGFuZCAocHJvYnM+PTApLmFsbCgpIGFuZCAocHJvYnM8PTEpLmFsbCgpCiBhc3NlcnQgbnAuYWxsY2xvc2UocHJvYnMuc3VtKDEpLDEsYXRvbD0yZS02LHJ0b2w9MCkgYW5kIG5wLmFycmF5X2VxdWFsKHByb2JzLmFyZ21heCgxKSxwKQogZ29sZD1wcm9ic1tucC5hcmFuZ2UobGVuKHkpKSx5XTtwb3NpdGl2ZT1nb2xkPjAKIGFzc2VydCBucC5hbGxjbG9zZSgtbnAubG9nKGdvbGRbcG9zaXRpdmVdKSxjZVtwb3NpdGl2ZV0scnRvbD0yZS01LGF0b2w9MmUtNSkKIGFzc2VydCAoY2VbfnBvc2l0aXZlXT49LW5wLmxvZyhucC5maW5mbyhucC5mbG9hdDMyKS50aW55KSkuYWxsKCkKIG1ldHJpY3M9eydhY2N1cmFjeSc6ZmxvYXQoYWNjdXJhY3lfc2NvcmUoeSxwKSksJ21pY3JvX2YxJzpmbG9hdChmMV9zY29yZSh5LHAsYXZlcmFnZT0nbWljcm8nKSksJ21hY3JvX2YxJzpmbG9hdChmMV9zY29yZSh5LHAsYXZlcmFnZT0nbWFjcm8nLGxhYmVscz1yYW5nZSg2KSx6ZXJvX2RpdmlzaW9uPTApKSwnd2VpZ2h0ZWRfZjEnOmZsb2F0KGYxX3Njb3JlKHkscCxhdmVyYWdlPSd3ZWlnaHRlZCcsbGFiZWxzPXJhbmdlKDYpLHplcm9fZGl2aXNpb249MCkpLCdiYWxhbmNlZF9hY2N1cmFjeSc6ZmxvYXQoYmFsYW5jZWRfYWNjdXJhY3lfc2NvcmUoeSxwKSksJ2xvc3MnOmZsb2F0KGNlLm1lYW4oKSksJ3NhbXBsZXMnOmxlbihmcmFtZSl9CiBkZWYgY29tcGFyZShhLGIpOgogIGlmIGlzaW5zdGFuY2UoYSxkaWN0KToKICAgYXNzZXJ0IHNldChhKT09c2V0KGIpCiAgIGZvciBrIGluIGE6Y29tcGFyZShhW2tdLGJba10pCiAgZWxzZTphc3NlcnQgYWJzKGEtYik8MWUtMTAKIGNvbXBhcmUoanNvbi5sb2FkcygoZm9sZGVyLyhwcmVmaXgrJ19tZXRyaWNzLmpzb24nKSkucmVhZF90ZXh0KCkpLG1ldHJpY3MpCiBjb21wYXJlKGpzb24ubG9hZHMoKGZvbGRlci8ocHJlZml4KydfY2xhc3NpZmljYXRpb25fcmVwb3J0Lmpzb24nKSkucmVhZF90ZXh0KCkpLGNsYXNzaWZpY2F0aW9uX3JlcG9ydCh5LHAsbGFiZWxzPXJhbmdlKDYpLHRhcmdldF9uYW1lcz1MQUJFTFMsb3V0cHV0X2RpY3Q9VHJ1ZSx6ZXJvX2RpdmlzaW9uPTApKQogY209Y29uZnVzaW9uX21hdHJpeCh5LHAsbGFiZWxzPXJhbmdlKDYpKTtjb25mPWpzb24ubG9hZHMoKGZvbGRlci8ocHJlZml4KydfY29uZnVzaW9ucy5qc29uJykpLnJlYWRfdGV4dCgpKTthc3NlcnQgY29uZlsnbGFiZWxzJ109PUxBQkVMUyBhbmQgbnAuYXJyYXlfZXF1YWwoY29uZlsncmF3J10sY20pCiBub3JtPW5wLmRpdmlkZShjbSxjbS5zdW0oMSxrZWVwZGltcz1UcnVlKSxvdXQ9bnAuemVyb3NfbGlrZShjbSxkdHlwZT1mbG9hdCksd2hlcmU9Y20uc3VtKDEsa2VlcGRpbXM9VHJ1ZSkhPTApCiBhc3NlcnQgbnAuYWxsY2xvc2UoY29uZlsncm93X25vcm1hbGl6ZWQnXSxub3JtLGF0b2w9MWUtMTIscnRvbD0wKQogcmV0dXJuIHsnc3RhdHVzJzoncGFzc2VkX2luZGVwZW5kZW50X2Nzdl92ZXJpZmljYXRpb24nLCdzcGxpdCc6c3BsaXQsJ21ldHJpY3MnOm1ldHJpY3MsJ3ByZWRpY3Rpb25fc2hhMjU2JzpzaGEoZm9sZGVyLyhwcmVmaXgrJ19wcmVkaWN0aW9ucy5jc3YnKSksJ2V4YWN0X29yZGVyX2lkc19sYWJlbHMnOlRydWUsJ3Byb2JhYmlsaXR5X2NlX2NvbmZ1c2lvbl9jaGVja3MnOlRydWV9Cg==', 'testing/text_plus_image/bert_deit3/evaluate_frozen.py': 'IiIiRXZhbHVhdGUgb25lIGZyb3plbiwgdHJhaW5pbmctY29tcGxldGUgY2hlY2twb2ludDsgbmV2ZXIgb3B0aW1pemUgb3Igc2VsZWN0IG9uIHRlc3QuIiIiCmltcG9ydCBhcmdwYXJzZSxkYXRldGltZSxoYXNobGliLGpzb24sb3Msc3lzLHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoClJPT1Q9UGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbM10Kc3lzLnBhdGguaW5zZXJ0KDAsc3RyKFJPT1QgLyAibW9kZWwvdGV4dF9wbHVzX2ltYWdlL2JlcnRfZGVpdDMiKSkKc3lzLnBhdGguaW5zZXJ0KDAsc3RyKFJPT1QgLyAidHJhaW5pbmcvdGV4dF9wbHVzX2ltYWdlL1NjcmlwdHMiKSkKc3lzLnBhdGguaW5zZXJ0KDAsc3RyKFJPT1QgLyAidGVzdGluZy90ZXh0X3BsdXNfaW1hZ2UiKSkKCmRlZiBub3coKTpyZXR1cm4gZGF0ZXRpbWUuZGF0ZXRpbWUubm93KGRhdGV0aW1lLnRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCkKZGVmIHJ1bihhKToKIGltcG9ydCBudW1weSBhcyBucCxwYW5kYXMgYXMgcGQsdG9yY2gKIGltcG9ydCB0cmFpbiBhcyB0CiBmcm9tIHVzYWJsZV9jb2hvcnRfdjIgaW1wb3J0IGxvYWRfaW5wdXQKIGZyb20gbXVsdGltb2RhbF9yZWNvdmVyeV92MSBpbXBvcnQgc2hhLGF0b21pY19qc29uLGxvYWRfZ2VuZXJhdGlvbgogZnJvbSB2ZXJpZnlfbXVsdGltb2RhbF9maW5hbF92MSBpbXBvcnQgdmVyaWZ5LExBQkVMUwogZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGNsYXNzaWZpY2F0aW9uX3JlcG9ydCxjb25mdXNpb25fbWF0cml4CiBmb2xkZXI9UGF0aChhLnJ1bik7cHJvdG9jb2w9anNvbi5sb2FkcygoZm9sZGVyLydwcm90b2NvbC5qc29uJykucmVhZF90ZXh0KCkpO2NmZz1wcm90b2NvbFsnY29uZmlnJ10KIHBheWxvYWQscHJvb2Y9bG9hZF9nZW5lcmF0aW9uKGZvbGRlci8ncmVjb3ZlcnknKTtzdGF0ZT1wYXlsb2FkWydzdGF0ZSddO2Fzc2VydCBwYXlsb2FkWydjb250cmFjdCddPT1wcm90b2NvbAogYXNzZXJ0IHN0YXRlWydlcG9jaCddPjQgb3Igc3RhdGVbJ2Vhcmx5X3N0b3BwZWQnXTthc3NlcnQgbGVuKHN0YXRlWydoaXN0b3J5J10pPj0zCiBzZWxlY3RlZD1tYXgoc3RhdGVbJ2hpc3RvcnknXSxrZXk9bGFtYmRhIGg6KGhbJ3ZhbGlkYXRpb24nXVsnYWNjdXJhY3knXSxoWyd2YWxpZGF0aW9uJ11bJ21hY3JvX2YxJ10sLWhbJ3ZhbGlkYXRpb24nXVsnbG9zcyddLC1oWydlcG9jaCddKSkKIGFzc2VydCBzZWxlY3RlZFsnZXBvY2gnXT09c3RhdGVbJ2Jlc3QnXVsnZXBvY2gnXTtiZXN0PXN0YXRlWydiZXN0J107d2VpZ2h0cz1mb2xkZXIvYmVzdFsnd2VpZ2h0c19maWxlJ107YXNzZXJ0IHNoYSh3ZWlnaHRzKT09YmVzdFsnd2VpZ2h0c19zaGEyNTYnXQogZnJlZXplPWZvbGRlci8nZmluYWxfc2VsZWN0aW9uX2Zyb3plbi5qc29uJwogdmFsdWU9eyd1dGMnOm5vdygpLCdzdGF0ZSc6J2Zyb3plbl9iZWZvcmVfdGVzdCcsJ3NlbGVjdGlvbl9ydWxlJzpjZmdbJ3NlbGVjdGlvbiddLCdzZWxlY3RlZCc6YmVzdCwncHJvdG9jb2xfc2hhMjU2JzpzaGEoZm9sZGVyLydwcm90b2NvbC5qc29uJyksJ2NvbmZpZ19zaGEyNTYnOnByb3RvY29sWydjb25maWdfc2hhMjU2J10sJ3NvdXJjZV9oYXNoZXMnOnByb3RvY29sWydzb3VyY2VzJ10sJ2lucHV0cyc6cHJvdG9jb2xbJ2lucHV0cyddLCd0cmFpbmluZ19jb21wbGV0ZSc6VHJ1ZSwncmVjb3ZlcnknOnByb29mfQogaWYgZnJlZXplLmV4aXN0cygpOgogIGV4aXN0aW5nPWpzb24ubG9hZHMoZnJlZXplLnJlYWRfdGV4dCgpKTthc3NlcnQgZXhpc3RpbmdbJ3NlbGVjdGVkJ109PWJlc3QgYW5kIGV4aXN0aW5nWydwcm90b2NvbF9zaGEyNTYnXT09dmFsdWVbJ3Byb3RvY29sX3NoYTI1NiddCiBlbHNlOmF0b21pY19qc29uKGZyZWV6ZSx2YWx1ZSkKIGV4cGVjdGVkLF89bG9hZF9pbnB1dChhLnJvb3QpO29mZmljaWFsPXBkLkRhdGFGcmFtZShbciBmb3IgciBpbiBleHBlY3RlZC52YWx1ZXMoKSBpZiByWydzcGxpdCddPT0ndGVzdCddKTthc3NlcnQgbGVuKG9mZmljaWFsKT09NTkyNzAKIG1hbmlmZXN0PXBkLnJlYWRfY3N2KFBhdGgoYS5yb290KS8nZGF0YS9vZmZpY2lhbF9wYWlyZWRfdjIvcGFpcmVkX21hbmlmZXN0LmNzdicsZHR5cGU9c3RyLGtlZXBfZGVmYXVsdF9uYT1GYWxzZSk7YXNzZXJ0IHNoYShQYXRoKGEucm9vdCkvJ2RhdGEvb2ZmaWNpYWxfcGFpcmVkX3YyL3BhaXJlZF9tYW5pZmVzdC5jc3YnKT09cHJvdG9jb2xbJ2lucHV0cyddWydtYW5pZmVzdF9zaGEyNTYnXQogdGVzdD1tYW5pZmVzdFttYW5pZmVzdC5zcGxpdC5lcSgndGVzdCcpXS5zZXRfaW5kZXgoJ2lkJykubG9jW29mZmljaWFsLmlkXS5yZXNldF9pbmRleCgpCiBmb3Iga2V5IGluIFsnY2xlYW5fdGl0bGUnLCc2X3dheV9sYWJlbCddOmFzc2VydCBucC5hcnJheV9lcXVhbCh0ZXN0W2tleV0uYXN0eXBlKHN0cikudG9fbnVtcHkoKSxvZmZpY2lhbFtrZXldLmFzdHlwZShzdHIpLnRvX251bXB5KCkpCiBpZiAoZm9sZGVyLyd0ZXN0X21ldHJpY3MuanNvbicpLmV4aXN0cygpOgogIHN0YXJ0PWpzb24ubG9hZHMoKGZvbGRlci8ndGVzdF9zdGFydC5qc29uJykucmVhZF90ZXh0KCkpCiAgYXNzZXJ0IHN0YXJ0Wyd3ZWlnaHRzX3NoYTI1NiddPT1iZXN0Wyd3ZWlnaHRzX3NoYTI1NiddIGFuZCBzdGFydFsnc2VsZWN0aW9uX3JlY2VpcHRfc2hhMjU2J109PXNoYShmcmVlemUpCiAgIyBSZWNvbnN0cnVjdCByZXBvcnRzIG9ubHkgZnJvbSB0aGUgYWxyZWFkeSBzYXZlZCBwcmVkaWN0aW9ucy4gTmV2ZXIgcmVydW4gaW5mZXJlbmNlLgogIHNhdmVkPXBkLnJlYWRfY3N2KGZvbGRlci8ndGVzdF9wcmVkaWN0aW9ucy5jc3YnLGR0eXBlPXsnaWQnOnN0cn0pCiAgeT1zYXZlZC50cnVlX2xhYmVsLnRvX251bXB5KCk7cHJlZD1zYXZlZC5wcmVkaWN0ZWRfbGFiZWwudG9fbnVtcHkoKTtjbT1jb25mdXNpb25fbWF0cml4KHkscHJlZCxsYWJlbHM9cmFuZ2UoNikpCiAgYXRvbWljX2pzb24oZm9sZGVyLyd0ZXN0X2NsYXNzaWZpY2F0aW9uX3JlcG9ydC5qc29uJyxjbGFzc2lmaWNhdGlvbl9yZXBvcnQoeSxwcmVkLGxhYmVscz1yYW5nZSg2KSx0YXJnZXRfbmFtZXM9TEFCRUxTLHplcm9fZGl2aXNpb249MCxvdXRwdXRfZGljdD1UcnVlKSkKICBub3JtPW5wLmRpdmlkZShjbSxjbS5zdW0oMSxrZWVwZGltcz1UcnVlKSxvdXQ9bnAuemVyb3NfbGlrZShjbSxkdHlwZT1mbG9hdCksd2hlcmU9Y20uc3VtKDEsa2VlcGRpbXM9VHJ1ZSkhPTApCiAgYXRvbWljX2pzb24oZm9sZGVyLyd0ZXN0X2NvbmZ1c2lvbnMuanNvbicseydsYWJlbHMnOkxBQkVMUywncmF3JzpjbS50b2xpc3QoKSwncm93X25vcm1hbGl6ZWQnOm5vcm0udG9saXN0KCl9KQogIHJlc3VsdD12ZXJpZnkoZm9sZGVyLCd0ZXN0JyxvZmZpY2lhbCwndGVzdCcpO2F0b21pY19qc29uKGZvbGRlci8ndGVzdF9pbmRlcGVuZGVudF92ZXJpZmljYXRpb24uanNvbicscmVzdWx0KQogIHByZWZpeD0ndmFsaWRhdGlvbl9lcG9jaCcrc3RyKGJlc3RbJ2Vwb2NoJ10pO3ZmPXBkLnJlYWRfY3N2KGZvbGRlci8ocHJlZml4KydfcHJlZGljdGlvbnMuY3N2JyksZHR5cGU9eydpZCc6c3RyfSk7dmYuaW5zZXJ0KDEsJ3NwbGl0JywndmFsaWRhdGlvbicpO3ZmLnRvX2Nzdihmb2xkZXIvJ3ZhbGlkYXRpb25fcHJlZGljdGlvbnMuY3N2JyxpbmRleD1GYWxzZSxmbG9hdF9mb3JtYXQ9JyUuMTdnJykKICBpbXBvcnQgc2h1dGlsCiAgZm9yIHN1ZmZpeCBpbiBbJ21ldHJpY3MuanNvbicsJ2NsYXNzaWZpY2F0aW9uX3JlcG9ydC5qc29uJywnY29uZnVzaW9ucy5qc29uJ106c2h1dGlsLmNvcHlmaWxlKGZvbGRlci8ocHJlZml4KydfJytzdWZmaXgpLGZvbGRlci8oJ3ZhbGlkYXRpb25fJytzdWZmaXgpKQogIHZvZmY9cGQuRGF0YUZyYW1lKFtyIGZvciByIGluIGV4cGVjdGVkLnZhbHVlcygpIGlmIHJbJ3NwbGl0J109PSd2YWxpZGF0aW9uJ10pO2F0b21pY19qc29uKGZvbGRlci8ndmFsaWRhdGlvbl9pbmRlcGVuZGVudF92ZXJpZmljYXRpb24uanNvbicsdmVyaWZ5KGZvbGRlciwndmFsaWRhdGlvbicsdm9mZiwndmFsaWRhdGlvbicpKQogIGF0b21pY19qc29uKGZvbGRlci8nZmluYWxfZXZhbHVhdGlvbl9jb21wbGV0ZS5qc29uJyx7J3V0Yyc6bm93KCksJ3Rlc3QnOnJlc3VsdCwnZmluYWxfc2VsZWN0ZWRfZXBvY2gnOmJlc3RbJ2Vwb2NoJ10sJ3N0YXRlJzoncnVudGltZV92ZXJpZmllZF9yZXF1aXJlc19zYXZlZF9wcml2YXRlX2FuZF9sb2NhbF9yZWFkYmFjayd9KQogIHJldHVybiByZXN1bHQKIGFzc2VydCBub3QgKGZvbGRlci8ndGVzdF9zdGFydC5qc29uJykuZXhpc3RzKCksJ0ludGVycnVwdGVkIHRlc3QgcmVxdWlyZXMgZXhwbGljaXRseSByZXZpZXdlZCByZXBsYXk7IG5ldmVyIHNpbGVudGx5IHJlLXJ1bicKIHRva2VuaXplcj10LkF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGZvbGRlci8ndG9rZW5pemVyJyk7YmNmZz10LkF1dG9Db25maWcuZnJvbV9wcmV0cmFpbmVkKFJPT1QvJ21vZGVsL3RleHRfcGx1c19pbWFnZS9iZXJ0X2RlaXQzL2JlcnQtYmFzZS1hc3NldHMnKQogaW1wb3J0IHRpbW0KIGltYWdlPXRpbW0uY3JlYXRlX21vZGVsKCdkZWl0M19iYXNlX3BhdGNoMTZfMjI0LmZiX2luMjJrX2Z0X2luMWsnLHByZXRyYWluZWQ9RmFsc2UpO2ltYWdlLnJlc2V0X2NsYXNzaWZpZXIoMCkKIG1vZGVsPXQuTWF4aW11bUZ1c2lvbih0LkJlcnRNb2RlbChiY2ZnLGFkZF9wb29saW5nX2xheWVyPUZhbHNlKSxpbWFnZSkKIGxvYWRlZD10b3JjaC5sb2FkKHdlaWdodHMsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1UcnVlKTttb2RlbC5sb2FkX3N0YXRlX2RpY3QobG9hZGVkLHN0cmljdD1UcnVlKTttb2RlbD1tb2RlbC5jdWRhKCkuZXZhbCgpCiAjIFNhbWUgc2VsZWN0ZWQgdGVuc29yLCB2YWxpZGF0aW9uLW9ubHkgcHJvYmU6IHN0cmljdCByZWxvYWQgcm91bmR0cmlwIGFuZCBsb2dpdHMuCiB2YWw9bWFuaWZlc3RbbWFuaWZlc3Quc3BsaXQuZXEoJ3ZhbGlkYXRpb24nKV0uaWxvY1s6MzJdLmNvcHkoKTtkcz10LlBhaXJzKHZhbCxhLnJvb3QsdG9rZW5pemVyLGNmZyxGYWxzZSxhLmluZGV4LHByb3RvY29sWydpbnB1dHMnXVsncGFja2VkX2luZGV4X3NoYTI1NiddKTtiYXRjaD1uZXh0KGl0ZXIodC5idWlsZF9sb2FkZXIoZHMsdG9rZW5pemVyLGNmZyxGYWxzZSkpKTtiYXRjaC5wb3AoJ2lkcycpO2JhdGNoLnBvcCgnbGFiZWxzJykKIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKSx0b3JjaC5hdXRvY2FzdCgnY3VkYScsZHR5cGU9dG9yY2guZmxvYXQxNik6YmVmb3JlPW1vZGVsKCoqe2s6di5jdWRhKCkgZm9yIGssdiBpbiBiYXRjaC5pdGVtcygpfSkKIG1vZGVsLmxvYWRfc3RhdGVfZGljdChsb2FkZWQsc3RyaWN0PVRydWUpCiB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCksdG9yY2guYXV0b2Nhc3QoJ2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MTYpOmFmdGVyPW1vZGVsKCoqe2s6di5jdWRhKCkgZm9yIGssdiBpbiBiYXRjaC5pdGVtcygpfSkKIGFzc2VydCB0b3JjaC5lcXVhbChiZWZvcmUsYWZ0ZXIpO2F0b21pY19qc29uKGZvbGRlci8nc2VsZWN0ZWRfd2VpZ2h0X3JvdW5kdHJpcC5qc29uJyx7J3V0Yyc6bm93KCksJ3NoYTI1Nic6c2hhKHdlaWdodHMpLCdzdHJpY3RfbG9hZCc6VHJ1ZSwndmFsaWRhdGlvbl9wcm9iZV9yb3dzJzpsZW4odmFsKSwnbWF4aW11bV9sb2dpdF9kZWx0YSc6ZmxvYXQoKGJlZm9yZS1hZnRlcikuYWJzKCkubWF4KCkpfSkKIGRlbCBsb2FkZWQsZHMsYmF0Y2gsYmVmb3JlLGFmdGVyCiBkYXRhPXQuUGFpcnModGVzdCxhLnJvb3QsdG9rZW5pemVyLGNmZyxGYWxzZSxhLmluZGV4LHByb3RvY29sWydpbnB1dHMnXVsncGFja2VkX2luZGV4X3NoYTI1NiddKTt0ZXN0X2xvYWRlcj10LmJ1aWxkX2xvYWRlcihkYXRhLHRva2VuaXplcixjZmcsRmFsc2UpCiBhdG9taWNfanNvbihmb2xkZXIvJ3Rlc3Rfc3RhcnQuanNvbicseyd1dGMnOm5vdygpLCd3ZWlnaHRzX3NoYTI1Nic6YmVzdFsnd2VpZ2h0c19zaGEyNTYnXSwnc2VsZWN0aW9uX3JlY2VpcHRfc2hhMjU2JzpzaGEoZnJlZXplKSwnZXhwZWN0ZWRfcm93cyc6NTkyNzAsJ3BvbGljeSc6J29uZSBmdWxsIGZyb3plbiBldmFsdWF0aW9uOyBubyBvcHRpbWl6ZXIgb3IgdHVuaW5nJ30pCiB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCk7c3RhcnRlZD10aW1lLm1vbm90b25pYygpO21ldHJpY3MsZnJhbWU9dC5ldmFsdWF0ZShtb2RlbCx0ZXN0X2xvYWRlcix0b3JjaC5kZXZpY2UoJ2N1ZGEnKSk7aW5mZXJlbmNlX2VsYXBzZWQ9dGltZS5tb25vdG9uaWMoKS1zdGFydGVkO2Fzc2VydCBmcmFtZS5pZC50b2xpc3QoKT09b2ZmaWNpYWwuaWQudG9saXN0KCk7ZnJhbWUuaW5zZXJ0KDEsJ3NwbGl0JywndGVzdCcpCiBmcmFtZS50b19jc3YoZm9sZGVyLyd0ZXN0X3ByZWRpY3Rpb25zLmNzdicsaW5kZXg9RmFsc2UsZmxvYXRfZm9ybWF0PSclLjE3ZycpO2F0b21pY19qc29uKGZvbGRlci8ndGVzdF9tZXRyaWNzLmpzb24nLG1ldHJpY3MpCiB5PWZyYW1lLnRydWVfbGFiZWwudG9fbnVtcHkoKTtwcmVkPWZyYW1lLnByZWRpY3RlZF9sYWJlbC50b19udW1weSgpO2NtPWNvbmZ1c2lvbl9tYXRyaXgoeSxwcmVkLGxhYmVscz1yYW5nZSg2KSk7YXRvbWljX2pzb24oZm9sZGVyLyd0ZXN0X2NsYXNzaWZpY2F0aW9uX3JlcG9ydC5qc29uJyxjbGFzc2lmaWNhdGlvbl9yZXBvcnQoeSxwcmVkLGxhYmVscz1yYW5nZSg2KSx0YXJnZXRfbmFtZXM9TEFCRUxTLHplcm9fZGl2aXNpb249MCxvdXRwdXRfZGljdD1UcnVlKSk7YXRvbWljX2pzb24oZm9sZGVyLyd0ZXN0X2NvbmZ1c2lvbnMuanNvbicseydsYWJlbHMnOkxBQkVMUywncmF3JzpjbS50b2xpc3QoKSwncm93X25vcm1hbGl6ZWQnOihjbS9jbS5zdW0oMSxrZWVwZGltcz1UcnVlKSkudG9saXN0KCl9KQogcmVzdWx0PXZlcmlmeShmb2xkZXIsJ3Rlc3QnLG9mZmljaWFsLCd0ZXN0Jyk7cmVzdWx0LnVwZGF0ZSh1dGM9bm93KCksaW5mZXJlbmNlX3NlY29uZHM9aW5mZXJlbmNlX2VsYXBzZWQsZXZhbHVhdGlvbl9leHBvcnRfdmVyaWZpY2F0aW9uX3NlY29uZHM9dGltZS5tb25vdG9uaWMoKS1zdGFydGVkLHBlYWtfdGVzdF9hbGxvY2F0ZWRfYnl0ZXM9dG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpLGdwdT10b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSxwYXJhbWV0ZXJzPXN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKSxzZWxlY3RlZF9lcG9jaD1iZXN0WydlcG9jaCddLHdlaWdodHNfc2hhMjU2PWJlc3RbJ3dlaWdodHNfc2hhMjU2J10pO2F0b21pY19qc29uKGZvbGRlci8ndGVzdF9pbmRlcGVuZGVudF92ZXJpZmljYXRpb24uanNvbicscmVzdWx0KQogIyBQcmVzZXJ2ZSBvcmlnaW5hbCBzZWxlY3RlZCB2YWxpZGF0aW9uIGZpbGU7IG1ha2UgYSBsYWJlbGxlZCBmaW5hbCBleHBvcnQgY29weS4KIHByZWZpeD0ndmFsaWRhdGlvbl9lcG9jaCcrc3RyKGJlc3RbJ2Vwb2NoJ10pO3ZmPXBkLnJlYWRfY3N2KGZvbGRlci8ocHJlZml4KydfcHJlZGljdGlvbnMuY3N2JyksZHR5cGU9eydpZCc6c3RyfSk7dmYuaW5zZXJ0KDEsJ3NwbGl0JywndmFsaWRhdGlvbicpO3ZmLnRvX2Nzdihmb2xkZXIvJ3ZhbGlkYXRpb25fcHJlZGljdGlvbnMuY3N2JyxpbmRleD1GYWxzZSxmbG9hdF9mb3JtYXQ9JyUuMTdnJykKIGltcG9ydCBzaHV0aWwKIGZvciBzdWZmaXggaW4gWydtZXRyaWNzLmpzb24nLCdjbGFzc2lmaWNhdGlvbl9yZXBvcnQuanNvbicsJ2NvbmZ1c2lvbnMuanNvbiddOnNodXRpbC5jb3B5ZmlsZShmb2xkZXIvKHByZWZpeCsnXycrc3VmZml4KSxmb2xkZXIvKCd2YWxpZGF0aW9uXycrc3VmZml4KSkKIHZvZmY9cGQuRGF0YUZyYW1lKFtyIGZvciByIGluIGV4cGVjdGVkLnZhbHVlcygpIGlmIHJbJ3NwbGl0J109PSd2YWxpZGF0aW9uJ10pO2F0b21pY19qc29uKGZvbGRlci8ndmFsaWRhdGlvbl9pbmRlcGVuZGVudF92ZXJpZmljYXRpb24uanNvbicsdmVyaWZ5KGZvbGRlciwndmFsaWRhdGlvbicsdm9mZiwndmFsaWRhdGlvbicpKQogYXRvbWljX2pzb24oZm9sZGVyLydmaW5hbF9ldmFsdWF0aW9uX2NvbXBsZXRlLmpzb24nLHsndXRjJzpub3coKSwndGVzdCc6cmVzdWx0LCdmaW5hbF9zZWxlY3RlZF9lcG9jaCc6YmVzdFsnZXBvY2gnXSwnc3RhdGUnOidydW50aW1lX3ZlcmlmaWVkX3JlcXVpcmVzX3NhdmVkX3ByaXZhdGVfYW5kX2xvY2FsX3JlYWRiYWNrJ30pCiByZXR1cm4gcmVzdWx0CmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpO3AuYWRkX2FyZ3VtZW50KCctLXJ1bicscmVxdWlyZWQ9VHJ1ZSx0eXBlPVBhdGgpO3AuYWRkX2FyZ3VtZW50KCctLXJvb3QnLHJlcXVpcmVkPVRydWUsdHlwZT1QYXRoKTtwLmFkZF9hcmd1bWVudCgnLS1pbmRleCcscmVxdWlyZWQ9VHJ1ZSx0eXBlPVBhdGgpO2E9cC5wYXJzZV9hcmdzKCk7cHJpbnQoanNvbi5kdW1wcyhydW4oYSkpLGZsdXNoPVRydWUpCg=='}
for name, encoded in files.items():
    p=source/name; p.parent.mkdir(parents=True,exist_ok=True); p.write_bytes(base64.b64decode(encoded))
run_candidates=[p.parent for p in Path('/kaggle/input').rglob('selected_epoch4_7dd15875227c43fcbd41d2d9929045ef.pt') if p.parent.name=='urop_bert_deit3_training_retry1']
assert len(run_candidates)==1, f'expected exactly one saved training output, found {run_candidates}'
source_run=run_candidates[0]
receipt=json.loads((source_run/'session_receipt.json').read_text())
assert receipt['state']=='stage_complete' and receipt['history_epochs']==4 and receipt['no_test'] is True
assert receipt['best']['epoch']==4 and receipt['best']['weights_sha256']=='94eb92621c7dcdf21feb5e7e7117e9c4b3a9fac75f9bf6b9e572024dab3c957a'
run=Path('/kaggle/working/urop_bert_deit3_training_retry1')
assert not run.exists(), 'refuse to overwrite evaluation output'
shutil.copytree(source_run,run)
manifest_candidates=list(Path('/kaggle/input').rglob('paired_manifest.csv'))
manifest_candidates=[p for p in manifest_candidates if p.parent.name=='official_paired_v2']
assert len(manifest_candidates)==1, f'expected one official manifest: {manifest_candidates}'
root=manifest_candidates[0].parents[2]
index_candidates=list(Path('/kaggle/input').rglob('fc8dc03a4f773da5762891c8392375add5fb8087251b527f3b37a880308e6018.sqlite'))
assert len(index_candidates)==1, f'expected the single verified packed index cache: {index_candidates}'
index=index_candidates[0]
assert hashlib.sha256(index.read_bytes()).hexdigest()=='fc8dc03a4f773da5762891c8392375add5fb8087251b527f3b37a880308e6018'
print('FROZEN_SELECTION_PRECHECK',json.dumps({'selected_checkpoint_sha256':hashlib.sha256((run/'selected_epoch4_7dd15875227c43fcbd41d2d9929045ef.pt').read_bytes()).hexdigest(),'run':str(run),'data_root':str(root),'index':str(index)}),flush=True)
cmd=[sys.executable,'-u',str(source/'testing/text_plus_image/bert_deit3/evaluate_frozen.py'),'--run',str(run),'--root',str(root),'--index',str(index)]
subprocess.run(cmd,check=True,env={**__import__('os').environ,'CUDA_VISIBLE_DEVICES':'0','TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1'})
print('ONE_HELD_OUT_EVALUATION_COMPLETE', (run/'test_metrics.json').read_text(),flush=True)
